# MAIN CODE

In [1]:
# =========================================
# INSTALL REQUIRED R PACKAGES (run once)
# =========================================

from rpy2.robjects import r

r('''
options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!requireNamespace("hoopR", quietly = TRUE)) {
    install.packages("hoopR")
}

if (!requireNamespace("dplyr", quietly = TRUE)) {
    install.packages("dplyr")
}

suppressPackageStartupMessages({
    library(hoopR)
    library(dplyr)
})
''')

print("✅ hoopR installed and loaded")

✅ hoopR installed and loaded


In [2]:
# -*- coding: utf-8 -*-
"""
NCAA_BB_v3.py — Self-Sufficient NCAABB Prediction Notebook
============================================================
Improvements over v2:
  1. Auto-retraining scheduler (daily for current season data refresh,
     weekly for full model refit) — runs automatically on startup.
  2. Injury CSV import using same directory convention as odds CSVs.
  3. Injury impact features: roster-weighted minutes-lost score per team.
  4. Elo rating system: per-team Elo updated each game, fed into model.
  5. Home-court calibration: neutral-site flag properly weighted.
  6. Ensemble model: XGBoost margin + XGBoost win + LightGBM stacking layer.
  7. Calibration: Isotonic regression calibration on win probability.
  8. Feature importance pruning: auto-drops near-zero importance features.
  9. Opponent-adjusted rolling stats (simple SRS-style).
 10. Rest-days feature: computed from schedule gaps.
 11. Retraining metadata: stores last-trained timestamps in JSON.
 12. All paths unified under BASE_DIR config block at the top.

USAGE (Colab):
  1. Run the SETUP cell to mount Drive and install packages.
  2. Run CONFIG to set your paths.
  3. Run everything — the scheduler will handle retraining automatically.
"""

# ============================================================
# CELL 1: SETUP — Install packages + Mount Drive
# ============================================================

# Uncomment for Colab:
# !apt-get -qq update
# !apt-get -qq install -y r-base
# !pip -q install rpy2 pyarrow pandas lightgbm scikit-learn xgboost ipywidgets

from google.colab import drive
drive.mount("/content/drive")

import os, sys, re, glob, json, warnings, contextlib, unicodedata, hashlib
from pathlib import Path
from datetime import datetime, timedelta, date

import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ============================================================
# CELL 2: CONFIG — All paths live here
# ============================================================

CURRENT_SEASON = 2026

BASE_DIR        = "/content/drive/MyDrive/NCAABB"
RAW_DIR         = os.path.join(BASE_DIR, "raw_hoopr_parquets")
ROTOWIRE_DIR    = os.path.join(BASE_DIR, "roto-odds")
INJURY_DIR      = os.path.join(BASE_DIR, "cbb-injuries")
MODEL_DIR       = os.path.join(BASE_DIR, "models")
METADATA_PATH   = os.path.join(MODEL_DIR, "retrain_metadata.json")
TEAM_BOX_DIR    = os.path.join(RAW_DIR, "team_box")
SCHEDULE_DIR    = os.path.join(RAW_DIR, "schedule")

# Retraining schedule
RETRAIN_DATA_HOURS   = 24    # re-pull current-season data every 24h
RETRAIN_MODEL_DAYS   = 7     # full model refit every 7 days

HIST_SEASONS    = list(range(2020, CURRENT_SEASON + 1))
ROLL_WINDOW     = 10          # games for rolling stats
ELO_K           = 20          # Elo K-factor
ELO_INIT        = 1500        # starting Elo

for d in [RAW_DIR, ROTOWIRE_DIR, INJURY_DIR, MODEL_DIR, TEAM_BOX_DIR, SCHEDULE_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Paths configured")

# ============================================================
# CELL 3: UTILITIES
# ============================================================

@contextlib.contextmanager
def suppress_stdout_stderr():
    with open(os.devnull, "w") as devnull:
        old_out, old_err = sys.stdout, sys.stderr
        try:
            sys.stdout = sys.stderr = devnull
            yield
        finally:
            sys.stdout, sys.stderr = old_out, old_err


def _load_metadata() -> dict:
    if os.path.exists(METADATA_PATH):
        try:
            with open(METADATA_PATH) as f:
                return json.load(f)
        except Exception:
            pass
    return {}


def _save_metadata(meta: dict):
    with open(METADATA_PATH, "w") as f:
        json.dump(meta, f, indent=2, default=str)


def _hours_since(ts_str) -> float:
    """Hours since ISO timestamp string. Returns inf if missing."""
    if not ts_str:
        return float("inf")
    try:
        dt = datetime.fromisoformat(str(ts_str))
        return (datetime.utcnow() - dt).total_seconds() / 3600
    except Exception:
        return float("inf")


def _days_since(ts_str) -> float:
    return _hours_since(ts_str) / 24


# ============================================================
# CELL 4: TEAM ALIAS + CANONICAL NAME
# ============================================================

TEAM_ALIASES = {
    "e michigan": "eastern michigan", "w michigan": "western michigan",
    "c michigan": "central michigan", "n illinois": "northern illinois",
    "n kentucky": "northern kentucky", "w carolina": "western carolina",
    "e kentucky": "eastern kentucky", "s illinois": "southern illinois",
    "n arizona": "northern arizona", "n colorado": "northern colorado",
    "e illinois": "eastern illinois", "w illinois": "western illinois",
    "c connecticut": "central connecticut", "e texas am": "east texas a&m",
    "e washington": "eastern washington", "e tennessee state": "east tennessee state",
    "c arkansas": "central arkansas", "oklahoma st": "oklahoma state",
    "florida st": "florida state", "arkansas st": "arkansas state",
    "colorado st": "colorado state", "arizona st": "arizona state",
    "kansas st": "kansas state", "oregon st": "oregon state",
    "san diego st": "san diego state", "boise st": "boise state",
    "morehead st": "morehead state", "tennessee st": "tennessee state",
    "south carolina st": "south carolina state", "sc state": "south carolina state",
    "n dakota st": "north dakota state", "north dakota st": "north dakota state",
    "s dakota st": "south dakota state", "south dakota st": "south dakota state",
    "tarleton st": "tarleton state", "delaware st": "delaware state",
    "jackson st": "jackson state", "alcorn st": "alcorn state",
    "miss valley st": "mississippi valley st", "morgan state": "morgan st",
    "georgia st": "georgia state", "nc state": "north carolina state",
    "n c state": "north carolina state", "mcneese": "mcneese state",
    "mcneese st": "mcneese state", "nwestern st": "northwestern state",
    "northwestern st": "northwestern state", "st johns": "st johns",
    "st john's": "st johns", "saint johns": "st johns",
    "st josephs": "saint josephs", "st marys": "saint marys",
    "st marys (cal)": "saint marys", "st peters": "saint peters",
    "st peter's": "saint peters", "st. peters": "saint peters",
    "saint francis": "st francis pa", "st francis": "st francis pa",
    "st francis (pa)": "st francis pa", "st thomas (mn)": "st thomas mn",
    "st. thomas (mn)": "st thomas mn", "uconn": "connecticut",
    "u conn": "connecticut", "ucf": "central florida",
    "uic": "illinois chicago", "illinois-chicago": "illinois chicago",
    "csu northridge": "cal state northridge", "csu fullerton": "cal state fullerton",
    "fullerton": "cal state fullerton", "bakersfield": "cal state bakersfield",
    "ca baptist": "california baptist", "cal poly": "cal poly san luis obispo",
    "santa barbara": "cal santa barbara", "utsa": "ut san antonio",
    "ut rio grande": "texas rio grande valley", "ut martin": "tennessee martin",
    "ut arlington": "texas arlington", "uta": "texas arlington",
    "east texas am": "east texas a&m", "texas am cc": "texas a&m corpus christi",
    "texas am corpus christi": "texas a&m corpus christi",
    "miami oh": "miami (oh)", "g washington": "george washington",
    "umass": "massachusetts", "pitt": "pittsburgh", "byu": "brigham young",
    "ga southern": "georgia southern", "ole miss": "mississippi",
    "so indiana": "southern indiana", "southern miss": "southern mississippi",
    "nicholls": "nicholls state", "sf austin": "stephen f austin",
    "hou christian": "houston christian", "pennsylvania": "penn",
    "seattle": "seattle u", "washington st": "washington state",
    "abilene chrstn": "abilene christian", "charleston": "college of charleston",
    "charleston so": "charleston southern", "bethune": "bethune cookman",
    "ualbany": "albany", "umbc": "maryland baltimore county",
    "fdu": "fairleigh dickinson", "long island": "liu brooklyn",
    "western ky": "western kentucky", "mtsu": "middle tennessee state",
    "little rock": "arkansas little rock", "se missouri": "se missouri state",
    "fiu": "florida international", "fgcu": "florida gulf coast",
    "florida a&m": "florida am", "jax state": "jacksonville state",
    "nc a&t": "north carolina a&t", "nc central": "north carolina central",
    "md eastern": "maryland eastern shore", "ar pine bluff": "arkansas pine bluff",
    "detroit": "detroit mercy", "siue": "southern illinois edwardsville",
    "sc upstate": "usc upstate", "loyola md": "loyola maryland",
    "loyola (md)": "loyola maryland", "lmu": "loyola marymount",
    "purdue fw": "purdue fort wayne", "purdue ft wayne": "purdue fort wayne",
    "ipfw": "purdue fort wayne", "etsu": "east tennessee state",
    "east tennessee st": "east tennessee state", "umass lowell": "massachusetts lowell",
    "green bay": "wisconsin green bay", "fau": "florida atlantic",
    "wichita st": "wichita state", "texas a&m": "texas am",
    "texas a and m": "texas am", "lsu": "louisiana state",
    "app state": "appalachian state", "app st": "appalachian state",
    "omaha": "nebraska omaha", "neb omaha": "nebraska omaha",
    "ga southern": "georgia southern", "coastal": "coastal carolina",
    "long beach st": "long beach state", "hawai'i": "hawaii", "hawaiʻi": "hawaii",
    "la salle": "la salle", "boston u": "boston university",
    "citadel": "the citadel", "nc state": "north carolina state",
    "gardner webb": "gardner webb", "queens university": "queens",
    "fiu": "florida international",
    # Aliases seen in real Rotowire odds file
    "unc-wilmington": "nc wilmington",
    "unc-charlotte": "charlotte",
    "unc-greensboro": "nc greensboro",
    "e. tennessee state": "east tennessee state",
    "e. tennessee st": "east tennessee state",
    "wisconsin-green bay": "wisconsin green bay",
    "texas-san antonio": "ut san antonio",
    "queens university": "queens",
    "william & mary": "william mary",
    "william and mary": "william mary",
    "illinois-chicago": "illinois chicago",
    "idaho state": "idaho state",
    "portland state": "portland state",
    "northern iowa": "northern iowa",
    "north dakota state": "north dakota state",
    "north dakota": "north dakota",
    "montana state": "montana state",
    "san francisco": "san francisco",
}


def canonical_team(x: str) -> str:
    s = str(x) if x is not None else ""
    s = s.strip().lower()
    s = "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))
    s = re.sub(r"[-/]", " ", s)
    s = re.sub(r"[^\w\s\(\)]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"^the\s+", "", s).strip()
    s = TEAM_ALIASES.get(s, s)
    s = re.sub(r"\sst$", " state", s)
    s = re.sub(r"\sst\)$", " state)", s)
    s = TEAM_ALIASES.get(s, s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


# ============================================================
# CELL 5: ELO RATING ENGINE
# FULL REWRITE — fixes merge_asof datetime dtype mismatch
# ============================================================

def compute_elo_ratings(team_box_all: pd.DataFrame, k: float = ELO_K, init: float = ELO_INIT) -> pd.DataFrame:
    """
    Computes per-team Elo ratings from chronological game history.

    Notes:
    - Processes each game once, not once per team row
    - Returns one row per team per game
    - Normalizes game_dt_et to datetime64[ns] to avoid merge_asof dtype issues
    """
    tb = team_box_all.copy()
    tb["game_id"] = tb["game_id"].astype(str)
    tb["game_date_time"] = pd.to_datetime(tb["game_date_time"], errors="coerce", utc=True)
    tb["team_id"] = pd.to_numeric(tb["team_id"], errors="coerce")
    tb["opponent_team_id"] = pd.to_numeric(tb["opponent_team_id"], errors="coerce")
    tb["team_score"] = pd.to_numeric(tb["team_score"], errors="coerce")
    tb["opponent_team_score"] = pd.to_numeric(tb["opponent_team_score"], errors="coerce")

    tb = tb.dropna(
        subset=["game_id", "game_date_time", "team_id", "opponent_team_id", "team_score", "opponent_team_score"]
    ).copy()

    tb["team_id"] = tb["team_id"].astype(int)
    tb["opponent_team_id"] = tb["opponent_team_id"].astype(int)

    one_game = (
        tb.sort_values(["game_date_time", "game_id"])
          .groupby("game_id", as_index=False)
          .first()
          .copy()
    )

    elo = {}
    rows = []

    for _, row in one_game.sort_values(["game_date_time", "game_id"]).iterrows():
        game_id = str(row["game_id"])
        game_dt = row["game_date_time"]

        t1 = int(row["team_id"])
        t2 = int(row["opponent_team_id"])
        s1 = float(row["team_score"])
        s2 = float(row["opponent_team_score"])

        r1 = elo.get(t1, init)
        r2 = elo.get(t2, init)

        e1 = 1.0 / (1.0 + 10 ** ((r2 - r1) / 400.0))
        e2 = 1.0 - e1

        if s1 > s2:
            a1, a2 = 1.0, 0.0
        elif s1 < s2:
            a1, a2 = 0.0, 1.0
        else:
            a1, a2 = 0.5, 0.5

        mov = abs(s1 - s2)
        mov_mult = np.log1p(mov) / np.log(2)
        mov_mult = min(mov_mult, 2.5)

        new_r1 = r1 + k * mov_mult * (a1 - e1)
        new_r2 = r2 + k * mov_mult * (a2 - e2)

        rows.append({
            "game_id": game_id,
            "team_id": t1,
            "game_dt_et": game_dt,
            "elo_before": r1,
            "elo_after": new_r1,
        })
        rows.append({
            "game_id": game_id,
            "team_id": t2,
            "game_dt_et": game_dt,
            "elo_before": r2,
            "elo_after": new_r2,
        })

        elo[t1] = new_r1
        elo[t2] = new_r2

    elo_df = pd.DataFrame(rows)

    if len(elo_df):
        elo_df["game_dt_et"] = (
            pd.to_datetime(elo_df["game_dt_et"], errors="coerce", utc=True)
              .dt.tz_convert("America/New_York")
              .dt.tz_localize(None)
              .astype("datetime64[ns]")
        )

    return elo_df


def get_latest_elo_snapshot(elo_df: pd.DataFrame) -> pd.DataFrame:
    if elo_df is None or len(elo_df) == 0:
        return pd.DataFrame(columns=["team_id", "elo_rating"])

    snap = (
        elo_df.sort_values(["team_id", "game_dt_et"])
              .groupby("team_id", as_index=False)
              .last()[["team_id", "elo_after"]]
              .rename(columns={"elo_after": "elo_rating"})
    )
    return snap


def add_elo_asof_features(joined: pd.DataFrame, elo_df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds leakage-safe as-of Elo values for home/away teams at game time.

    Critical fix:
    - both sides of merge_asof are forced to datetime64[ns]
    """
    base = joined.copy()

    if elo_df is None or len(elo_df) == 0 or len(base) == 0:
        base["elo_home"] = float(ELO_INIT)
        base["elo_away"] = float(ELO_INIT)
        base["elo_teamA"] = float(ELO_INIT)
        base["elo_teamB"] = float(ELO_INIT)
        base["diff_elo"] = 0.0
        return base

    base["game_dt_et"] = pd.to_datetime(base["game_dt_et"], errors="coerce").astype("datetime64[ns]")

    elo = elo_df.copy()
    elo["team_id"] = pd.to_numeric(elo["team_id"], errors="coerce")
    elo["game_dt_et"] = pd.to_datetime(elo["game_dt_et"], errors="coerce").astype("datetime64[ns]")
    elo = elo.dropna(subset=["team_id", "game_dt_et", "elo_after"]).copy()
    elo["team_id"] = elo["team_id"].astype(int)

    right = (
        elo[["team_id", "game_dt_et", "elo_after"]]
        .sort_values(["game_dt_et", "team_id"], kind="mergesort")
        .reset_index(drop=True)
    )

    def _merge_side(df: pd.DataFrame, team_col: str) -> pd.Series:
        left = pd.DataFrame({
            "row_id": np.arange(len(df), dtype=int),
            "team_id": pd.to_numeric(df[team_col], errors="coerce"),
            "game_dt_et": pd.to_datetime(df["game_dt_et"], errors="coerce").astype("datetime64[ns]"),
        }).dropna().copy()

        out = pd.Series(float(ELO_INIT), index=df.index, dtype=float)

        if len(left) == 0:
            return out

        left["team_id"] = left["team_id"].astype(int)
        left = left.sort_values(["game_dt_et", "team_id"], kind="mergesort").reset_index(drop=True)

        merged = pd.merge_asof(
            left,
            right,
            on="game_dt_et",
            by="team_id",
            direction="backward",
            allow_exact_matches=False,
        ).sort_values("row_id")

        out.iloc[merged["row_id"].values] = merged["elo_after"].fillna(float(ELO_INIT)).values
        return out

    base["elo_home"] = _merge_side(base, "home_id")
    base["elo_away"] = _merge_side(base, "away_id")

    teamA_is_home = (base["teamA_id"].astype(int) == base["home_id"].astype(int)).values
    base["elo_teamA"] = np.where(teamA_is_home, base["elo_home"], base["elo_away"])
    base["elo_teamB"] = np.where(teamA_is_home, base["elo_away"], base["elo_home"])
    base["diff_elo"] = base["elo_teamA"] - base["elo_teamB"]

    return base


# ============================================================
# CELL 6: INJURY LOADER
# ============================================================
# File format (college-basketball-injury-report-MMDD.xlsx):
#   Columns: Player, Team, Pos, Injury, Status, Est. Return
#   Status values: "Out", "Game Time Decision", "Doubtful", "Probable"
#
# Naming convention mirrors odds files:
#   college-basketball-injury-report-MMDD.xlsx   (primary)
#   college-basketball-injury-report-MMDD{a,b}.xlsx (updates)
#   Also accepts .csv variants with same columns.
# ============================================================

def find_injury_files_for_date(date_et, injury_dir: str) -> list:
    """
    Finds injury files for a given date.
    Supports:
      college-basketball-injury-report-MMDD.xlsx / .csv
      cbb-injuries-MMDD.xlsx / .csv
      cbb-injuries-YYYY-MM-DD.xlsx / .csv
    Returns files sorted: base first, then a/b/c suffix updates.
    """
    d = pd.Timestamp(date_et).date()
    mmdd     = f"{d.month:02d}{d.day:02d}"
    yyyymmdd = f"{d.year}-{d.month:02d}-{d.day:02d}"

    patterns = [
        re.compile(rf"college-basketball-injury-report-{mmdd}([a-z]?)\.(xlsx|csv)$", re.IGNORECASE),
        re.compile(rf"cbb-injuries-{mmdd}([a-z]?)\.(xlsx|csv)$", re.IGNORECASE),
        re.compile(rf"cbb-injuries-{yyyymmdd}([a-z]?)\.(xlsx|csv)$", re.IGNORECASE),
    ]

    hits = []
    for f in glob.glob(os.path.join(injury_dir, "*")):
        bn = os.path.basename(f)
        for pat in patterns:
            m = pat.search(bn)
            if m:
                suffix = (m.group(1) or "").lower()
                hits.append((suffix, f))
                break

    hits.sort(key=lambda x: (x[0] != "", x[0]))  # base first, then a, b, c...
    return [f for _, f in hits]


def _parse_injury_file(path: str) -> pd.DataFrame:
    """
    Parses a single injury file (xlsx or csv).

    Expected columns (exact from Rotowire xlsx):
      Player, Team, Pos, Injury, Status, Est. Return

    Status values observed:
      "Out"                  → out (full weight)
      "Game Time Decision"   → questionable (50% weight)
      "Doubtful"             → out (90% weight — treated as out)
      "Probable"             → active (10% weight — ignored)

    Minutes per game: NOT in the file, so we use a position-based
    proxy (C/F=28, G=26, default=25) which is a reasonable NCAABB average.
    If the file ever gains an MPG column it will be used automatically.
    """
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext in {".xlsx", ".xls"}:
            df = pd.read_excel(path, dtype=str)
        else:
            df = pd.read_csv(path, dtype=str)
    except Exception as e:
        print(f"⚠️ Could not read injury file {os.path.basename(path)}: {e}")
        return pd.DataFrame()

    if len(df) == 0:
        return pd.DataFrame()

    # Normalize column names
    df.columns = [str(c).strip() for c in df.columns]
    col_map = {c.lower(): c for c in df.columns}

    def _get(variants):
        for v in variants:
            if v in col_map:
                return col_map[v]
        return None

    player_col = _get(["player", "player_name", "name"])
    team_col   = _get(["team", "teams", "school"])
    pos_col    = _get(["pos", "position"])
    status_col = _get(["status", "injury_status", "game_status"])
    inj_col    = _get(["injury", "injury_type", "description"])
    mpg_col    = _get(["mpg", "minutes_per_game", "min_per_game", "avg_minutes", "minutes"])

    if player_col is None or team_col is None or status_col is None:
        print(f"⚠️ Injury file missing required columns. Found: {list(df.columns)}")
        return pd.DataFrame()

    out = pd.DataFrame()
    out["player"] = df[player_col].astype(str).str.strip()
    out["team"]   = df[team_col].astype(str).str.strip()
    out["pos"]    = df[pos_col].astype(str).str.strip() if pos_col else "?"
    out["status_raw"] = df[status_col].astype(str).str.strip()
    out["injury"] = df[inj_col].astype(str).str.strip() if inj_col else ""

    # MPG: use file value if present, otherwise position proxy
    if mpg_col:
        out["minutes_per_game"] = pd.to_numeric(df[mpg_col], errors="coerce")
    else:
        # Position-based proxy for NCAABB (reasonable default)
        _pos_mpg = {"c": 24, "f": 26, "g": 26, "g/f": 26, "f/c": 25}
        out["minutes_per_game"] = out["pos"].str.lower().map(_pos_mpg).fillna(25.0)

    # Status bucketing — matches real Rotowire values
    def _bucket(s: str) -> str:
        s = str(s).strip().lower()
        if s in {"out", "doubtful", "d", "dnp", "suspended", "out for season"}:
            return "out"
        if s in {"game time decision", "gtd", "questionable", "q", "day-to-day",
                 "dtd", "probable", "p", "50/50"}:
            return "questionable"
        return "active"

    out["status_bucket"] = out["status_raw"].map(_bucket)

    # Impact weight: out=1.0, questionable=0.5, active=0.0
    _weight = {"out": 1.0, "questionable": 0.5, "active": 0.0}
    out["impact_weight"] = out["status_bucket"].map(_weight)

    # Canonical team name for matching
    out["team_canon"] = out["team"].map(canonical_team)

    # Drop rows with no meaningful impact
    out = out[out["status_bucket"] != "active"].copy()

    return out.reset_index(drop=True)


def load_injury_data_for_date(date_et, injury_dir: str) -> pd.DataFrame:
    """Loads and merges all injury files for a given slate date."""
    files = find_injury_files_for_date(date_et, injury_dir)
    if not files:
        return pd.DataFrame()

    dfs = []
    for f in files:
        parsed = _parse_injury_file(f)
        if len(parsed) > 0:
            parsed["source_file"] = os.path.basename(f)
            dfs.append(parsed)

    if not dfs:
        return pd.DataFrame()

    combined = pd.concat(dfs, ignore_index=True)
    # If multiple files for same date (a/b updates), keep the latest entry per player
    combined = combined.drop_duplicates(subset=["team_canon", "player"], keep="last")
    return combined


def compute_injury_impact(injury_df: pd.DataFrame) -> pd.DataFrame:
    """
    Computes per-team injury impact score from parsed injury file.

    injury_impact_score = sum(minutes_per_game * impact_weight) per team
      where impact_weight = 1.0 for Out/Doubtful, 0.5 for Game Time Decision/Questionable

    Returns DataFrame: team_canon, injury_impact_score, injury_count_out, injury_count_q
    """
    if injury_df is None or len(injury_df) == 0:
        return pd.DataFrame(columns=["team_canon", "injury_impact_score",
                                     "injury_count_out", "injury_count_q"])

    df = injury_df.copy()
    df["minutes_per_game"] = pd.to_numeric(df["minutes_per_game"], errors="coerce").fillna(25.0)
    df["impact_weight"]    = pd.to_numeric(df["impact_weight"], errors="coerce").fillna(0.5)
    df["weighted_mpg"]     = df["minutes_per_game"] * df["impact_weight"]

    g = df.groupby("team_canon").apply(lambda d: pd.Series({
        "injury_impact_score": d["weighted_mpg"].sum(),
        "injury_count_out":    (d["status_bucket"] == "out").sum(),
        "injury_count_q":      (d["status_bucket"] == "questionable").sum(),
        "injured_players":     ", ".join(d["player"].dropna().tolist()),
    })).reset_index()

    return g


def attach_injury_features_to_board(board: pd.DataFrame, date_et, injury_dir: str) -> pd.DataFrame:
    """Merges injury impact features into board (home/away columns)."""
    inj_raw = load_injury_data_for_date(date_et, injury_dir)
    inj = compute_injury_impact(inj_raw)

    if len(inj) == 0:
        for col in ["home_injury_impact", "away_injury_impact", "diff_injury_impact",
                    "home_inj_out", "away_inj_out"]:
            board[col] = np.nan
        return board

    b = board.copy()
    b["k_home"] = b["home_team"].map(canonical_team)
    b["k_away"] = b["away_team"].map(canonical_team)

    inj_dict = inj.set_index("team_canon")["injury_impact_score"].to_dict()
    out_dict  = inj.set_index("team_canon")["injury_count_out"].to_dict()

    b["home_injury_impact"] = b["k_home"].map(inj_dict).fillna(0.0)
    b["away_injury_impact"] = b["k_away"].map(inj_dict).fillna(0.0)
    b["diff_injury_impact"] = b["home_injury_impact"] - b["away_injury_impact"]
    b["home_inj_out"]       = b["k_home"].map(out_dict).fillna(0.0)
    b["away_inj_out"]       = b["k_away"].map(out_dict).fillna(0.0)
    return b.drop(columns=["k_home", "k_away"], errors="ignore")


# ============================================================
# CELL 7: ROTOWIRE ODDS LOADER
# ============================================================
# Exact format of cbb-odds-rotowire-MMDD.csv:
#
#   Row 0: ",, Win,, Cover,, Total Points,"     ← skip (junk header)
#   Row 1: "Team, Date, Moneyline, $20 Bet, Spread, $20 Bet, Over-Under, $20 Bet"
#   Row 2+: one team per row, paired away then home:
#
#   Penn State, 2026-03-08 12:00:00, 180,  36.00,  5.5, 19.05, 149.5, 17.39  ← away
#   Rutgers,   2026-03-08 12:00:00, -200, 10.00, -4.5, 17.39, 150.5, 18.18  ← home
#
#   Spread column = home-team spread (home row, negative = home fav)
#   Moneyline = away ML from away row, home ML from home row
#   Total = Over-Under from home row (or away row as fallback)

RW_FIELDS = ["rw_away_ml", "rw_home_ml", "rw_spread_home", "rw_total"]


def _to_float(x):
    if pd.isna(x): return np.nan
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}: return np.nan
    s = s.replace("−", "-").replace("–", "-").replace("½", ".5")
    s2 = re.sub(r"[^0-9.\-]", "", s)
    try: return float(s2) if s2 not in ("", "-", ".") else np.nan
    except: return np.nan


def _parse_rotowire_csv(path: str) -> pd.DataFrame:
    """
    Parses a Rotowire NCAABB odds CSV.
    Skips the first junk header row; uses the second row as column names.
    Pairs rows as (away, home) to build one game per output row.
    """
    try:
        raw = pd.read_csv(path, header=None, dtype=str, encoding="utf-8-sig")
    except UnicodeDecodeError:
        try:
            raw = pd.read_csv(path, header=None, dtype=str, encoding="latin-1")
        except Exception as e:
            print(f"⚠️ Could not read {os.path.basename(path)}: {e}")
            return pd.DataFrame()
    except Exception as e:
        print(f"⚠️ Could not read {os.path.basename(path)}: {e}")
        return pd.DataFrame()

    if len(raw) < 3:
        return pd.DataFrame()

    # Find the real header row: must contain "Team" AND "Moneyline"
    header_idx = None
    for i, row in raw.iterrows():
        vals = [str(v).strip().lower() for v in row.values]
        if "team" in vals and any("moneyline" in v for v in vals):
            header_idx = i
            break

    if header_idx is None:
        header_idx = 1   # fallback: row index 1 (0-based)

    headers = [str(v).strip() for v in raw.iloc[header_idx].values]
    data    = raw.iloc[header_idx + 1:].copy().reset_index(drop=True)
    data.columns = headers

    # Locate columns by case-insensitive name
    col_lower = {c.strip().lower(): c for c in data.columns}

    def _find(keys):
        for k in keys:
            if k in col_lower:
                return col_lower[k]
        return None

    team_col   = _find(["team", "teams"])
    ml_col     = _find(["moneyline", "ml"])
    spread_col = _find(["spread"])
    total_col  = _find(["over-under", "over/under", "o/u", "ou", "total", "totals"])

    if team_col is None:
        team_col = data.columns[0]

    # Drop rows where team name is blank or a header artifact
    def _bad(s):
        s = str(s).strip().lower()
        return s in {"", "team", "teams", "nan", "none"} or re.fullmatch(r"[\d:apm /\-]+", s) is not None

    data = data[~data[team_col].map(_bad)].reset_index(drop=True)

    rows = []
    i = 0
    while i + 1 < len(data):
        r_away = data.iloc[i]
        r_home = data.iloc[i + 1]

        away_name = str(r_away[team_col]).strip()
        home_name = str(r_home[team_col]).strip()

        if _bad(away_name) or _bad(home_name):
            i += 1
            continue

        away_ml     = _to_float(r_away[ml_col])      if ml_col     else np.nan
        home_ml     = _to_float(r_home[ml_col])      if ml_col     else np.nan
        spread_home = _to_float(r_home[spread_col])  if spread_col else np.nan
        total       = _to_float(r_home[total_col])   if total_col  else np.nan
        if pd.isna(total) and total_col:
            total = _to_float(r_away[total_col])

        rows.append({
            "away_team":      away_name,
            "home_team":      home_name,
            "rw_away_ml":     away_ml,
            "rw_home_ml":     home_ml,
            "rw_spread_home": spread_home,
            "rw_total":       total,
        })
        i += 2

    return pd.DataFrame(rows)


def find_rotowire_files_for_date(date_et, rotowire_dir: str) -> list:
    d = pd.Timestamp(date_et).date()
    mmdd = f"{d.month:02d}{d.day:02d}"
    pat = re.compile(rf"cbb-odds-rotowire-{mmdd}([a-z]?)\.csv$", re.IGNORECASE)
    hits = []
    for f in glob.glob(os.path.join(rotowire_dir, "*.csv")):
        bn = os.path.basename(f)
        m = pat.search(bn)
        if m:
            suffix = (m.group(1) or "").lower()
            hits.append((suffix, f))
    hits.sort(key=lambda x: (x[0] != "", x[0]))
    return [f for _, f in hits]


def load_rotowire_all_for_date(date_et, rotowire_dir: str) -> pd.DataFrame:
    """Loads and concatenates all Rotowire files for a slate date."""
    files = find_rotowire_files_for_date(date_et, rotowire_dir)
    if not files:
        return pd.DataFrame()
    dfs = [_parse_rotowire_csv(f) for f in files]
    dfs = [d for d in dfs if len(d) > 0]
    if not dfs:
        return pd.DataFrame()
    out = pd.concat(dfs, ignore_index=True)
    out = out.dropna(subset=["away_team", "home_team"])
    out = out[out["away_team"].str.strip().ne("") & out["home_team"].str.strip().ne("")]
    return out.reset_index(drop=True)


# ============================================================
# CELL 8: DATA LOADING + FEATURE ENGINEERING
# FULL REWRITE — FIX TBD + ROTOWIRE MATCHUP COMPLETION
# ============================================================

def _coerce_bool_series(x, index=None) -> pd.Series:
    if isinstance(x, pd.Series):
        s = x.copy()
    else:
        if index is None:
            index = pd.RangeIndex(0)
        s = pd.Series(x, index=index)

    if s.dtype == bool:
        return s.fillna(False)

    return (
        s.astype(str)
         .str.strip()
         .str.lower()
         .isin(["true", "1", "yes", "y", "t"])
         .fillna(False)
    )


def _bad_team_name(x) -> bool:
    s = str(x).strip().lower()
    return s in {"", "nan", "none", "null", "tbd", "tba", "unknown"}


def _safe_team_text_col(df: pd.DataFrame, candidates: list, default="TBD") -> pd.Series:
    for c in candidates:
        if c in df.columns:
            return df[c].astype(str).fillna(default)
    return pd.Series([default] * len(df), index=df.index)


def _build_team_id_name_map(schedule_df: pd.DataFrame, team_snaps: pd.DataFrame = None) -> dict:
    """
    canonical team name -> team_id
    built from schedule first, then team_snaps if available
    """
    frames = []

    if schedule_df is not None and len(schedule_df) > 0:
        s = schedule_df.copy()

        if {"home_id", "home_short_display_name"}.issubset(s.columns):
            x = s[["home_id", "home_short_display_name"]].copy()
            x.columns = ["team_id", "team_name"]
            frames.append(x)

        if {"away_id", "away_short_display_name"}.issubset(s.columns):
            x = s[["away_id", "away_short_display_name"]].copy()
            x.columns = ["team_id", "team_name"]
            frames.append(x)

        if {"home_id", "home_team"}.issubset(s.columns):
            x = s[["home_id", "home_team"]].copy()
            x.columns = ["team_id", "team_name"]
            frames.append(x)

        if {"away_id", "away_team"}.issubset(s.columns):
            x = s[["away_id", "away_team"]].copy()
            x.columns = ["team_id", "team_name"]
            frames.append(x)

    if team_snaps is not None and len(team_snaps) > 0:
        ts = team_snaps.copy()
        for id_col in ["team_id"]:
            if id_col in ts.columns:
                for nm_col in ["team_name", "display_name", "short_display_name", "team"]:
                    if nm_col in ts.columns:
                        x = ts[[id_col, nm_col]].copy()
                        x.columns = ["team_id", "team_name"]
                        frames.append(x)

    if not frames:
        return {}

    out = pd.concat(frames, ignore_index=True)
    out["team_id"] = pd.to_numeric(out["team_id"], errors="coerce")
    out["team_name"] = out["team_name"].astype(str).str.strip()

    out = out.dropna(subset=["team_id"]).copy()
    out = out[~out["team_name"].map(_bad_team_name)].copy()
    out["team_id"] = out["team_id"].astype(int)
    out["team_canon"] = out["team_name"].map(canonical_team)

    out = out.drop_duplicates(subset=["team_canon"], keep="last")
    return dict(zip(out["team_canon"], out["team_id"]))


def _extract_valid_schedule_slate(schedule_df: pd.DataFrame, slate_date_et) -> pd.DataFrame:
    if schedule_df is None or len(schedule_df) == 0:
        return pd.DataFrame()

    s = schedule_df.copy()
    slate_date_et = pd.Timestamp(slate_date_et).date()

    dt_col = None
    for c in ["game_dt_et", "game_date_time", "start_date", "date"]:
        if c in s.columns:
            dt_col = c
            break

    if dt_col is None:
        return pd.DataFrame()

    s[dt_col] = pd.to_datetime(s[dt_col], errors="coerce", utc=True)

    if pd.api.types.is_datetime64tz_dtype(s[dt_col]):
        s["game_dt_et"] = s[dt_col].dt.tz_convert("America/New_York").dt.tz_localize(None)
    else:
        s["game_dt_et"] = pd.to_datetime(s[dt_col], errors="coerce")

    s = s.dropna(subset=["game_dt_et"]).copy()
    s["slate_date_et"] = s["game_dt_et"].dt.date
    s = s[s["slate_date_et"] == slate_date_et].copy()

    if len(s) == 0:
        return pd.DataFrame()

    # normalize ids
    for c in ["game_id", "home_id", "away_id", "home_score", "away_score"]:
        if c in s.columns:
            if c == "game_id":
                s[c] = s[c].astype(str)
            else:
                s[c] = pd.to_numeric(s[c], errors="coerce")

    if "neutral_site" in s.columns:
        s["neutral_site"] = _coerce_bool_series(s["neutral_site"], s.index)
    else:
        s["neutral_site"] = False

    if "home_short_display_name" not in s.columns:
        s["home_short_display_name"] = _safe_team_text_col(
            s, ["home_team", "home_display_name", "home_name"], default="TBD"
        )
    if "away_short_display_name" not in s.columns:
        s["away_short_display_name"] = _safe_team_text_col(
            s, ["away_team", "away_display_name", "away_name"], default="TBD"
        )

    if "home_team" not in s.columns:
        s["home_team"] = s["home_short_display_name"].astype(str)
    if "away_team" not in s.columns:
        s["away_team"] = s["away_short_display_name"].astype(str)

    req = ["home_id", "away_id"]
    for c in req:
        if c not in s.columns:
            s[c] = np.nan

    s = s.dropna(subset=["home_id", "away_id"]).copy()
    if len(s) == 0:
        return pd.DataFrame()

    s["home_id"] = s["home_id"].astype(int)
    s["away_id"] = s["away_id"].astype(int)

    if "game_id" in s.columns:
        s = s.sort_values(["game_dt_et", "game_id"])
        s = s.drop_duplicates(subset=["game_id"], keep="last")
    else:
        s = s.sort_values(["game_dt_et", "home_id", "away_id"])
        s = s.drop_duplicates(subset=["game_dt_et", "home_id", "away_id"], keep="last")

    return s.reset_index(drop=True)


def _build_rotowire_fallback_slate(
    slate_date_et,
    schedule_df: pd.DataFrame,
    rotowire_dir: str,
    team_snaps: pd.DataFrame = None,
) -> pd.DataFrame:
    rw = load_rotowire_all_for_date(slate_date_et, rotowire_dir)
    if rw is None or len(rw) == 0:
        return pd.DataFrame()

    name_to_id = _build_team_id_name_map(schedule_df, team_snaps)

    out = rw.copy()
    out["away_team"] = out["away_team"].astype(str).str.strip()
    out["home_team"] = out["home_team"].astype(str).str.strip()
    out["away_canon"] = out["away_team"].map(canonical_team)
    out["home_canon"] = out["home_team"].map(canonical_team)
    out["away_id"] = out["away_canon"].map(name_to_id)
    out["home_id"] = out["home_canon"].map(name_to_id)

    out = out.dropna(subset=["away_id", "home_id"]).copy()
    if len(out) == 0:
        return pd.DataFrame()

    out["away_id"] = out["away_id"].astype(int)
    out["home_id"] = out["home_id"].astype(int)

    base_dt = pd.Timestamp(slate_date_et).tz_localize("America/New_York").tz_localize(None)
    out["game_dt_et"] = base_dt + pd.to_timedelta(np.arange(len(out)) * 5, unit="m")
    out["game_id"] = [
        f"RW_{pd.Timestamp(slate_date_et).strftime('%Y%m%d')}_{i}_{a}_{h}"
        for i, (a, h) in enumerate(zip(out["away_id"], out["home_id"]))
    ]

    out["neutral_site"] = False
    out["away_short_display_name"] = out["away_team"]
    out["home_short_display_name"] = out["home_team"]
    out["home_score"] = np.nan
    out["away_score"] = np.nan
    out["status_type_completed"] = False
    out["status_type_state"] = "pre"
    out["status_type_name"] = "STATUS_SCHEDULED"
    out["status_type_short_detail"] = ""

    keep = [
        "game_id", "game_dt_et", "neutral_site",
        "home_id", "away_id",
        "home_team", "away_team",
        "home_short_display_name", "away_short_display_name",
        "home_score", "away_score",
        "status_type_completed", "status_type_state",
        "status_type_name", "status_type_short_detail"
    ]
    return out[keep].reset_index(drop=True)


def _fill_tbd_schedule_from_rotowire(
    slate: pd.DataFrame,
    slate_date_et,
    schedule_df: pd.DataFrame,
    rotowire_dir: str,
    team_snaps: pd.DataFrame = None,
) -> pd.DataFrame:
    """
    If schedule has TBD on one side, fill the missing side using Rotowire.
    Match priority:
      1) exact known home team canon
      2) exact known away team canon
      3) unique unused RW row
    """
    if slate is None or len(slate) == 0:
        return slate

    rw = load_rotowire_all_for_date(slate_date_et, rotowire_dir)
    if rw is None or len(rw) == 0:
        return slate

    name_to_id = _build_team_id_name_map(schedule_df, team_snaps)
    rw = rw.copy()
    rw["rw_idx"] = np.arange(len(rw))
    rw["home_team"] = rw["home_team"].astype(str).str.strip()
    rw["away_team"] = rw["away_team"].astype(str).str.strip()
    rw["home_canon"] = rw["home_team"].map(canonical_team)
    rw["away_canon"] = rw["away_team"].map(canonical_team)
    rw["home_id"] = rw["home_canon"].map(name_to_id)
    rw["away_id"] = rw["away_canon"].map(name_to_id)

    b = slate.copy()
    b["home_team"] = _safe_team_text_col(b, ["home_team", "home_short_display_name"], default="TBD")
    b["away_team"] = _safe_team_text_col(b, ["away_team", "away_short_display_name"], default="TBD")
    b["home_short_display_name"] = _safe_team_text_col(b, ["home_short_display_name", "home_team"], default="TBD")
    b["away_short_display_name"] = _safe_team_text_col(b, ["away_short_display_name", "away_team"], default="TBD")

    used_rw = set()

    for idx in b.index:
        home_name = str(b.at[idx, "home_team"]).strip()
        away_name = str(b.at[idx, "away_team"]).strip()
        home_bad = _bad_team_name(home_name)
        away_bad = _bad_team_name(away_name)

        if not (home_bad or away_bad):
            continue

        cand = rw.copy()

        # match on known side
        if not home_bad:
            hc = canonical_team(home_name)
            cand = cand[cand["home_canon"] == hc].copy()

        if not away_bad:
            ac = canonical_team(away_name)
            cand = cand[cand["away_canon"] == ac].copy()

        # if still multiple, prefer one not used yet
        if len(cand) > 1:
            cand = cand[~cand["rw_idx"].isin(used_rw)].copy()

        # weak fallback: if only one schedule row is unresolved and one rw row unused fits
        if len(cand) == 0:
            remaining = rw[~rw["rw_idx"].isin(used_rw)].copy()
            if len(remaining) == 1:
                cand = remaining

        if len(cand) == 0:
            continue

        row = cand.iloc[0]
        used_rw.add(int(row["rw_idx"]))

        # overwrite both sides from RW so the matchup is fully coherent
        b.at[idx, "home_team"] = row["home_team"]
        b.at[idx, "away_team"] = row["away_team"]
        b.at[idx, "home_short_display_name"] = row["home_team"]
        b.at[idx, "away_short_display_name"] = row["away_team"]

        if pd.notna(row.get("home_id")):
            b.at[idx, "home_id"] = int(row["home_id"])
        if pd.notna(row.get("away_id")):
            b.at[idx, "away_id"] = int(row["away_id"])

    return b


def ensure_board_team_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Guarantees home_team / away_team always exist.
    """
    if df is None:
        return pd.DataFrame()

    out = df.copy()

    if "home_team" not in out.columns:
        if "home_short_display_name" in out.columns:
            out["home_team"] = out["home_short_display_name"].astype(str)
        else:
            out["home_team"] = "TBD"

    if "away_team" not in out.columns:
        if "away_short_display_name" in out.columns:
            out["away_team"] = out["away_short_display_name"].astype(str)
        else:
            out["away_team"] = "TBD"

    out["home_team"] = out["home_team"].astype(str).replace({"nan": "TBD", "None": "TBD"})
    out["away_team"] = out["away_team"].astype(str).replace({"nan": "TBD", "None": "TBD"})

    if "home_short_display_name" not in out.columns:
        out["home_short_display_name"] = out["home_team"]
    if "away_short_display_name" not in out.columns:
        out["away_short_display_name"] = out["away_team"]

    return out


def build_pregame_dataset_for_slate(
    schedule_df: pd.DataFrame,
    team_snaps: pd.DataFrame,
    roll_cols: list,
    slate_date_et,
    window: int = ROLL_WINDOW,
    elo_snap: pd.DataFrame = None,
) -> pd.DataFrame:
    slate_date_et = pd.Timestamp(slate_date_et).date()

    slate = _extract_valid_schedule_slate(schedule_df, slate_date_et)

    # fill partial schedule rows like "TBD vs Prairie View" using Rotowire
    if slate is not None and len(slate) > 0:
        slate = _fill_tbd_schedule_from_rotowire(
            slate=slate,
            slate_date_et=slate_date_et,
            schedule_df=schedule_df,
            rotowire_dir=ROTOWIRE_DIR,
            team_snaps=team_snaps,
        )

    # if slate is still bad/incomplete, replace with full Rotowire slate
    if slate is None or len(slate) == 0:
        slate = _build_rotowire_fallback_slate(
            slate_date_et=slate_date_et,
            schedule_df=schedule_df,
            rotowire_dir=ROTOWIRE_DIR,
            team_snaps=team_snaps,
        )
    else:
        bad_mask = (
            slate["home_team"].map(_bad_team_name)
            | slate["away_team"].map(_bad_team_name)
            | slate["home_id"].isna()
            | slate["away_id"].isna()
        )
        if bad_mask.any():
            rw_full = _build_rotowire_fallback_slate(
                slate_date_et=slate_date_et,
                schedule_df=schedule_df,
                rotowire_dir=ROTOWIRE_DIR,
                team_snaps=team_snaps,
            )
            if rw_full is not None and len(rw_full) > 0:
                slate = rw_full

    if slate is None or len(slate) == 0:
        return pd.DataFrame()

    slate = ensure_board_team_columns(slate)

    snaps = team_snaps.copy()
    snaps["team_id"] = pd.to_numeric(snaps["team_id"], errors="coerce")
    snaps["game_dt_et"] = pd.to_datetime(snaps["game_dt_et"], errors="coerce")

    snaps = snaps.dropna(subset=["team_id", "game_dt_et"]).copy()
    snaps["team_id"] = snaps["team_id"].astype("int64")
    snaps["game_dt_et"] = snaps["game_dt_et"].astype("datetime64[ns]")

    snaps = snaps.sort_values(["game_dt_et", "team_id"], kind="mergesort").reset_index(drop=True)
    use_roll_cols = [c for c in roll_cols if c in snaps.columns]

    def asof_team(team_ids, times, prefix):
        out = pd.DataFrame(index=np.arange(len(team_ids)))

        left = pd.DataFrame({
            "row_id": np.arange(len(team_ids), dtype="int64"),
            "team_id": pd.to_numeric(pd.Series(team_ids), errors="coerce"),
            "game_dt_et": pd.to_datetime(pd.Series(times), errors="coerce"),
        })

        left = left.dropna(subset=["team_id", "game_dt_et"]).copy()
        if len(left) == 0 or len(use_roll_cols) == 0:
            for c in use_roll_cols:
                out[f"{c}_{prefix}"] = np.nan
            return out

        # force exact dtypes for merge_asof
        left["team_id"] = left["team_id"].astype("int64")
        left["game_dt_et"] = pd.to_datetime(left["game_dt_et"], errors="coerce").astype("datetime64[ns]")
        left = left.sort_values(["game_dt_et", "team_id"], kind="mergesort").reset_index(drop=True)

        right = snaps[["team_id", "game_dt_et"] + use_roll_cols].copy()
        right["team_id"] = pd.to_numeric(right["team_id"], errors="coerce")
        right["game_dt_et"] = pd.to_datetime(right["game_dt_et"], errors="coerce")

        right = right.dropna(subset=["team_id", "game_dt_et"]).copy()
        if len(right) == 0:
            for c in use_roll_cols:
                out[f"{c}_{prefix}"] = np.nan
            return out

        right["team_id"] = right["team_id"].astype("int64")
        right["game_dt_et"] = right["game_dt_et"].astype("datetime64[ns]")
        right = right.sort_values(["game_dt_et", "team_id"], kind="mergesort").reset_index(drop=True)

        merged = pd.merge_asof(
            left,
            right,
            on="game_dt_et",
            by="team_id",
            direction="backward",
            allow_exact_matches=True,
        ).sort_values("row_id")

        for c in use_roll_cols:
            out[f"{c}_{prefix}"] = np.nan
            if c in merged.columns:
                out.loc[merged["row_id"].to_numpy(), f"{c}_{prefix}"] = merged[c].to_numpy()

        return out

    home_feats = asof_team(slate["home_id"].values, slate["game_dt_et"].values, "HOME")
    away_feats = asof_team(slate["away_id"].values, slate["game_dt_et"].values, "AWAY")

    base_cols = [
        "game_id", "game_dt_et", "neutral_site",
        "home_id", "away_id",
        "home_team", "away_team",
        "home_short_display_name", "away_short_display_name",
        "home_score", "away_score",
        "status_type_completed", "status_type_state",
        "status_type_name", "status_type_short_detail",
    ]
    base_cols = [c for c in base_cols if c in slate.columns]
    joined = pd.concat([slate[base_cols].reset_index(drop=True), home_feats, away_feats], axis=1)

    joined = ensure_board_team_columns(joined)

    joined["home_id"] = pd.to_numeric(joined["home_id"], errors="coerce").astype(int)
    joined["away_id"] = pd.to_numeric(joined["away_id"], errors="coerce").astype(int)

    joined["teamA_id"] = np.minimum(joined["home_id"], joined["away_id"]).astype(int)
    joined["teamB_id"] = np.maximum(joined["home_id"], joined["away_id"]).astype(int)

    teamA_is_home = joined["teamA_id"].values == joined["home_id"].values

    TA, TB, DIFF = {}, {}, {}
    for c in use_roll_cols:
        h = f"{c}_HOME"
        a = f"{c}_AWAY"
        if h not in joined.columns or a not in joined.columns:
            continue

        ta = np.where(teamA_is_home, joined[h], joined[a])
        tb = np.where(teamA_is_home, joined[a], joined[h])

        TA[f"{c}_TA"] = ta
        TB[f"{c}_TB"] = tb
        DIFF[f"diff_{c}"] = ta - tb

    if DIFF:
        joined = pd.concat(
            [
                joined,
                pd.DataFrame(TA, index=joined.index),
                pd.DataFrame(TB, index=joined.index),
                pd.DataFrame(DIFF, index=joined.index),
            ],
            axis=1
        )

    diff_cols_local = [c for c in joined.columns if c.startswith("diff_")]
    if diff_cols_local:
        joined[diff_cols_local] = joined[diff_cols_local].apply(pd.to_numeric, errors="coerce")

    if elo_snap is not None and len(elo_snap) > 0:
        joined = add_elo_asof_features(joined, elo_snap)

    neutral = _coerce_bool_series(joined.get("neutral_site", False), joined.index)
    joined["home_court_A"] = np.where(neutral, 0.0, np.where(teamA_is_home, 1.0, -1.0))

    if joined.columns.duplicated().any():
        joined = joined.loc[:, ~joined.columns.duplicated()].copy()

    return ensure_board_team_columns(joined).reset_index(drop=True)

# ============================================================
# CELL 8b: SAFETY PATCH FOR DOWNSTREAM
# ============================================================
def normalize_board_for_downstream(board: pd.DataFrame) -> pd.DataFrame:
    b = ensure_board_team_columns(board)

    # keep these mirrored so old code doesn't break
    if "home_short_display_name" not in b.columns:
        b["home_short_display_name"] = b["home_team"]
    if "away_short_display_name" not in b.columns:
        b["away_short_display_name"] = b["away_team"]

    return b



# ============================================================
# CELL 9: TRAINING DATASET BUILDER (FULL REWRITE)
# ============================================================

def build_game_dataset_from_team_box(
    team_box: pd.DataFrame,
    schedule_df: pd.DataFrame,
    window: int = ROLL_WINDOW,
    elo_df: pd.DataFrame = None
) -> pd.DataFrame:
    """
    Builds the per-game labeled dataset for model training.
    TeamA = min(team_id, opponent_team_id), TeamB = max.
    y_margin = TeamA score - TeamB score.
    y_win    = 1 if TeamA wins.

    Includes:
    - rolling diff features
    - Elo diff
    - home_court_A
    """
    tb = team_box.copy()
    tb["game_id"] = tb["game_id"].astype(str)
    tb["game_date_time"] = pd.to_datetime(tb["game_date_time"], errors="coerce", utc=True)
    tb["team_id"] = pd.to_numeric(tb["team_id"], errors="coerce")
    tb["opponent_team_id"] = pd.to_numeric(tb["opponent_team_id"], errors="coerce")
    tb["team_score"] = pd.to_numeric(tb["team_score"], errors="coerce")
    tb["opp_score"] = pd.to_numeric(tb["opponent_team_score"], errors="coerce")

    tb = tb.dropna(subset=[
        "team_id", "opponent_team_id", "team_score", "opp_score", "game_date_time"
    ]).copy()

    tb["team_id"] = tb["team_id"].astype(int)
    tb["opponent_team_id"] = tb["opponent_team_id"].astype(int)
    tb["game_dt_et"] = tb["game_date_time"].dt.tz_convert("America/New_York").dt.tz_localize(None)

    exclude = {
        "game_id", "season", "season_type", "game_date", "game_date_time", "game_dt_et",
        "team_id", "team_uid", "opponent_team_id", "opponent_team_uid",
        "team_score", "opp_score", "opponent_team_score",
    }

    num_cols = [
        c for c in tb.columns
        if c not in exclude
        and not str(c).startswith("opponent_")
        and pd.api.types.is_numeric_dtype(tb[c])
    ]

    for c in num_cols:
        tb[c] = pd.to_numeric(tb[c], errors="coerce")

    tb = tb.sort_values(["team_id", "game_dt_et"]).copy()

    roll_data = {}
    for c in num_cols:
        roll_data[f"r{window}_mean_{c}"] = (
            tb.groupby("team_id")[c]
              .rolling(window, min_periods=3).mean()
              .shift(1).reset_index(level=0, drop=True)
        )
        roll_data[f"r3_mean_{c}"] = (
            tb.groupby("team_id")[c]
              .rolling(3, min_periods=2).mean()
              .shift(1).reset_index(level=0, drop=True)
        )
        roll_data[f"r{window}_std_{c}"] = (
            tb.groupby("team_id")[c]
              .rolling(window, min_periods=3).std()
              .shift(1).reset_index(level=0, drop=True)
        )

    roll_data[f"r{window}_gp"] = (
        tb.groupby("team_id")["team_id"]
          .rolling(window, min_periods=1).count()
          .shift(1).reset_index(level=0, drop=True)
    )

    for k, v in roll_data.items():
        tb[k] = v

    roll_cols = list(roll_data.keys())

    lo = np.minimum(tb["team_id"], tb["opponent_team_id"])
    hi = np.maximum(tb["team_id"], tb["opponent_team_id"])
    tb["teamA_id"] = lo
    tb["teamB_id"] = hi
    tb["teamA_is_team"] = (tb["team_id"] == tb["teamA_id"])

    A = tb[tb["teamA_is_team"]].copy()
    B = tb[~tb["teamA_is_team"]].copy()

    keep_A = ["game_id", "season", "game_dt_et", "teamA_id", "teamB_id", "team_score", "opp_score"] + roll_cols
    keep_B = ["game_id"] + roll_cols

    A = A[[c for c in keep_A if c in A.columns]].copy()
    B = B[[c for c in keep_B if c in B.columns]].copy()

    A = A.rename(columns={c: f"{c}_A" for c in roll_cols})
    B = B.rename(columns={c: f"{c}_B" for c in roll_cols})

    merged = A.merge(B, on="game_id", how="inner")

    merged["y_margin"] = merged["team_score"] - merged["opp_score"]
    merged["y_win"] = (merged["y_margin"] > 0).astype(int)

    diff_cols = []
    for c in roll_cols:
        ca, cb = f"{c}_A", f"{c}_B"
        if ca in merged.columns and cb in merged.columns:
            dc = f"diff_{c}"
            merged[dc] = merged[ca] - merged[cb]
            diff_cols.append(dc)

    if elo_df is not None and len(elo_df) > 0:
        elo_map = (
            elo_df.sort_values(["team_id", "game_dt_et"])
                  .drop_duplicates(subset=["game_id", "team_id"], keep="last")
                  .set_index(["game_id", "team_id"])["elo_before"]
        )

        def _get_elo(gid, tid, default=ELO_INIT):
            try:
                return float(elo_map.loc[(str(gid), int(tid))])
            except Exception:
                return default

        merged["elo_A"] = [_get_elo(r["game_id"], r["teamA_id"]) for _, r in merged.iterrows()]
        merged["elo_B"] = [_get_elo(r["game_id"], r["teamB_id"]) for _, r in merged.iterrows()]
        merged["diff_elo"] = merged["elo_A"] - merged["elo_B"]
        diff_cols.append("diff_elo")

    sched = schedule_df.copy()
    sched["game_id"] = sched["game_id"].astype(str)
    sched["home_id"] = pd.to_numeric(sched.get("home_id"), errors="coerce")
    sched["away_id"] = pd.to_numeric(sched.get("away_id"), errors="coerce")

    keep_sched = [c for c in ["game_id", "home_id", "away_id", "neutral_site"] if c in sched.columns]
    sched = sched[keep_sched].copy()

    merged = merged.merge(sched, on="game_id", how="left")

    neutral = merged.get("neutral_site", pd.Series(False, index=merged.index))
    if neutral.dtype == object:
        neutral = neutral.astype(str).str.lower().isin(["true", "1", "yes", "y"])

    teamA_is_home = merged["teamA_id"] == pd.to_numeric(merged["home_id"], errors="coerce")
    merged["home_court_A"] = np.where(neutral, 0, np.where(teamA_is_home, 1, -1)).astype(float)
    diff_cols.append("home_court_A")

    merged[diff_cols] = merged[diff_cols].fillna(0.0)

    keep = [
        "game_id", "season", "game_dt_et", "teamA_id", "teamB_id",
        "y_margin", "y_win"
    ] + diff_cols

    return merged[[c for c in keep if c in merged.columns]].copy()


# ============================================================
# CELL 10: MODEL TRAINING (FULL REWRITE)
# ============================================================

def season_weights(season_series, current_season=CURRENT_SEASON, half_life=2.0):
    age = (current_season - season_series).clip(lower=0).astype(float)
    return (0.5 ** (age / half_life)).values


def train_models(dataset_all: pd.DataFrame, current_season: int = CURRENT_SEASON):
    """
    Trains:
      1. XGBoost spread (margin regression)
      2. XGBoost win (classification)
      3. LightGBM margin
      4. Isotonic calibration on win probability
    """
    train = dataset_all[dataset_all["season"] < current_season].copy()
    current = dataset_all[dataset_all["season"] == current_season].copy()

    if len(train) < 100:
        raise ValueError("Not enough historical training data to fit models.")

    diff_cols = [c for c in dataset_all.columns if c.startswith("diff_") or c == "home_court_A"]
    diff_cols = [c for c in diff_cols if c in train.columns]

    for c in diff_cols:
        dataset_all[c + "__na"] = dataset_all[c].isna().astype("int8")
        train[c + "__na"] = train[c].isna().astype("int8")
        if len(current) > 0:
            current[c + "__na"] = current[c].isna().astype("int8")

    diff_cols2 = diff_cols + [c + "__na" for c in diff_cols]

    imp = SimpleImputer(strategy="median")
    train[diff_cols] = imp.fit_transform(train[diff_cols])
    if len(current) > 0:
        current[diff_cols] = imp.transform(current[diff_cols])

    var = train[diff_cols2].var()
    keep_cols = var[var > 1e-8].index.tolist()
    diff_cols2 = [c for c in diff_cols2 if c in keep_cols]

    print(f"Features after variance filter: {len(diff_cols2)}")

    train_sorted = train.sort_values("game_dt_et").copy()

    if len(train_sorted) < 500:
        split_n = max(1, int(len(train_sorted) * 0.85))
        tr_idx = train_sorted.iloc[:split_n].index
        va_idx = train_sorted.iloc[split_n:].index
    else:
        cut = train_sorted["game_dt_et"].max() - pd.Timedelta(days=30)
        tr_idx = train_sorted[train_sorted["game_dt_et"] < cut].index
        va_idx = train_sorted[train_sorted["game_dt_et"] >= cut].index

        if len(tr_idx) == 0 or len(va_idx) == 0:
            split_n = max(1, int(len(train_sorted) * 0.85))
            tr_idx = train_sorted.iloc[:split_n].index
            va_idx = train_sorted.iloc[split_n:].index

    X_tr = train.loc[tr_idx, diff_cols2].copy()
    y_tr_m = train.loc[tr_idx, "y_margin"].clip(-30, 30)
    y_tr_w = train.loc[tr_idx, "y_win"]
    w_tr = season_weights(train.loc[tr_idx, "season"], current_season)

    X_va = train.loc[va_idx, diff_cols2].copy()
    y_va_m = train.loc[va_idx, "y_margin"].clip(-30, 30)
    y_va_w = train.loc[va_idx, "y_win"]
    w_va = season_weights(train.loc[va_idx, "season"], current_season)

    if len(X_tr) == 0 or len(X_va) == 0:
        raise ValueError("Training/validation split is empty. Check dataset size and split logic.")

    dtrain = xgb.DMatrix(X_tr, label=y_tr_m, weight=w_tr, feature_names=diff_cols2)
    dvalid = xgb.DMatrix(X_va, label=y_va_m, weight=w_va, feature_names=diff_cols2)

    params_reg = {
        "objective": "reg:pseudohubererror",
        "eval_metric": "mae",
        "eta": 0.02,
        "max_depth": 4,
        "min_child_weight": 5,
        "subsample": 0.85,
        "colsample_bytree": 0.80,
        "lambda": 1.5,
        "alpha": 0.1,
        "seed": 42,
    }

    spread_booster = xgb.train(
        params=params_reg,
        dtrain=dtrain,
        num_boost_round=5000,
        evals=[(dvalid, "valid")],
        early_stopping_rounds=150,
        verbose_eval=False,
    )
    print(f"✅ XGB Margin best_iter: {spread_booster.best_iteration}")

    dtrain_w = xgb.DMatrix(X_tr, label=y_tr_w, weight=w_tr, feature_names=diff_cols2)
    dvalid_w = xgb.DMatrix(X_va, label=y_va_w, weight=w_va, feature_names=diff_cols2)

    params_clf = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "eta": 0.02,
        "max_depth": 3,
        "min_child_weight": 10,
        "subsample": 0.80,
        "colsample_bytree": 0.75,
        "lambda": 3.0,
        "alpha": 0.5,
        "seed": 42,
    }

    winner_booster = xgb.train(
        params=params_clf,
        dtrain=dtrain_w,
        num_boost_round=5000,
        evals=[(dvalid_w, "valid")],
        early_stopping_rounds=150,
        verbose_eval=False,
    )
    print(f"✅ XGB Win best_iter: {winner_booster.best_iteration}")

    lgb_train = lgb.Dataset(X_tr, label=y_tr_m, weight=w_tr, feature_name=diff_cols2)
    lgb_valid = lgb.Dataset(X_va, label=y_va_m, weight=w_va, reference=lgb_train, feature_name=diff_cols2)

    lgb_params = {
        "objective": "huber",
        "alpha": 0.9,
        "metric": "mae",
        "learning_rate": 0.02,
        "num_leaves": 31,
        "min_data_in_leaf": 20,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.85,
        "bagging_freq": 5,
        "lambda_l1": 0.1,
        "lambda_l2": 1.0,
        "verbose": -1,
        "seed": 42,
    }

    lgb_callbacks = [
        lgb.early_stopping(100, verbose=False),
        lgb.log_evaluation(period=-1),
    ]

    lgb_spread = lgb.train(
        lgb_params,
        lgb_train,
        num_boost_round=3000,
        valid_sets=[lgb_valid],
        callbacks=lgb_callbacks,
    )
    print(f"✅ LGB Margin best_iter: {lgb_spread.best_iteration}")

    raw_va_w = winner_booster.predict(
        dvalid_w,
        iteration_range=(0, winner_booster.best_iteration + 1)
    )
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(raw_va_w, y_va_w.values)
    print("✅ Isotonic calibration fitted")

    feature_cols = diff_cols2
    return spread_booster, winner_booster, lgb_spread, iso, imp, diff_cols2, feature_cols


def save_models(spread_booster, winner_booster, lgb_spread, iso, imp, model_dir: str):
    import pickle
    spread_booster.save_model(os.path.join(model_dir, "spread_booster.json"))
    winner_booster.save_model(os.path.join(model_dir, "winner_booster.json"))
    lgb_spread.save_model(os.path.join(model_dir, "lgb_spread.txt"))
    with open(os.path.join(model_dir, "iso_calibrator.pkl"), "wb") as f:
        pickle.dump(iso, f)
    with open(os.path.join(model_dir, "imputer.pkl"), "wb") as f:
        pickle.dump(imp, f)
    print(f"✅ Models saved to {model_dir}")


def load_models(model_dir: str):
    import pickle

    spread_booster = xgb.Booster()
    spread_booster.load_model(os.path.join(model_dir, "spread_booster.json"))

    winner_booster = xgb.Booster()
    winner_booster.load_model(os.path.join(model_dir, "winner_booster.json"))

    lgb_spread = lgb.Booster(model_file=os.path.join(model_dir, "lgb_spread.txt"))

    with open(os.path.join(model_dir, "iso_calibrator.pkl"), "rb") as f:
        iso = pickle.load(f)

    with open(os.path.join(model_dir, "imputer.pkl"), "rb") as f:
        imp = pickle.load(f)

    print("✅ Models loaded from disk")
    return spread_booster, winner_booster, lgb_spread, iso, imp



# ============================================================
# CELL 10A: R / hoopR DATA REFRESH PIPELINE
# MUST RUN BEFORE PYTHON LOADS team_box / schedule
# ============================================================

# One-time setup for Colab runtime
# !apt-get -qq update
# !apt-get -qq install -y r-base
# !pip -q install rpy2 pyarrow pandas

import os
from rpy2.robjects import r

def run_hoopr_refresh(
    raw_dir: str,
    current_season: int,
    seasons=None,
    force: bool = False,
    check_changes_for_current: bool = True,
    export=None,
):
    """
    Runs the user's hoopR -> parquet export pipeline in R.
    This updates schedule/team_box/etc before Python reloads them.
    """
    if seasons is None:
        seasons = list(range(2020, current_season + 1))

    if export is None:
        export = {
            "schedule": True,
            "team_box": True,
            "player_box": True,
            "pbp": True,
            "teams": True,
            "standings": True,
            "rankings": True,
            "team_stats": True,
            "player_stats": True,
        }

    os.makedirs(raw_dir, exist_ok=True)

    r.assign("RAW_DIR", raw_dir)
    r.assign("SEASONS", seasons)
    r.assign("FORCE", force)
    r.assign("CURRENT_SEASON", current_season)
    r.assign("CHECK_CHANGES_FOR_CURRENT", check_changes_for_current)

    r.assign("EXPORT_SCHEDULE", export["schedule"])
    r.assign("EXPORT_TEAM_BOX", export["team_box"])
    r.assign("EXPORT_PLAYER_BOX", export["player_box"])
    r.assign("EXPORT_PBP", export["pbp"])
    r.assign("EXPORT_TEAMS", export["teams"])
    r.assign("EXPORT_STANDINGS", export["standings"])
    r.assign("EXPORT_RANKINGS", export["rankings"])
    r.assign("EXPORT_TEAM_STATS", export["team_stats"])
    r.assign("EXPORT_PLAYER_STATS", export["player_stats"])

    r_code = r'''
    options(warn = -1)

    pkgs <- c("hoopR","dplyr","arrow","lubridate","data.table","stringr","progressr","digest")
    for (p in pkgs) {
      if (!requireNamespace(p, quietly = TRUE)) {
        install.packages(p, repos="https://cloud.r-project.org")
      }
      suppressPackageStartupMessages(library(p, character.only = TRUE))
    }

    if (!exists("CURRENT_SEASON")) CURRENT_SEASON <- max(SEASONS)
    CHECK_CURRENT_FOR_CHANGES <- CHECK_CHANGES_FOR_CURRENT
    CHECK_STATIC_FOR_CHANGES  <- FALSE

    dir.create(RAW_DIR, recursive=TRUE, showWarnings=FALSE)

    manifest_path <- file.path(RAW_DIR, "_manifest.csv")

    manifest <- NULL
    if (file.exists(manifest_path) && !FORCE) {
      manifest <- tryCatch(read.csv(manifest_path, stringsAsFactors = FALSE), error=function(e) NULL)
    }

    log_row <- function(dataset, season, season_type, path, status, message="", data_hash="") {
      row <- data.frame(
        ts = as.character(Sys.time()),
        dataset = dataset,
        season = ifelse(is.null(season), "", as.character(season)),
        season_type = ifelse(is.null(season_type), "", as.character(season_type)),
        path = path,
        status = status,
        data_hash = data_hash,
        message = message,
        stringsAsFactors = FALSE
      )
      if (is.null(manifest)) manifest <<- row else manifest <<- rbind(manifest, row)
      write.csv(manifest, manifest_path, row.names = FALSE)
    }

    clean_hash <- function(x) {
      if (length(x) == 0) return("")
      if (is.na(x)) return("")
      as.character(x)
    }

    last_ok_hash <- function(dataset, season=NULL, season_type=NULL, path=NULL) {
      if (is.null(manifest)) return("")
      m <- manifest
      cond <- m$dataset == dataset & m$status == "ok"
      if (!is.null(season))      cond <- cond & m$season == as.character(season)
      if (!is.null(season_type)) cond <- cond & m$season_type == as.character(season_type)
      if (!is.null(path))        cond <- cond & m$path == path
      if (!any(cond)) return("")
      idx <- which(cond)
      clean_hash(m$data_hash[idx[length(idx)]])
    }

    out_path <- function(rel) file.path(RAW_DIR, rel)

    write_one <- function(df, path) {
      dir.create(dirname(path), recursive=TRUE, showWarnings=FALSE)
      arrow::write_parquet(df, path)
      cat("✅ wrote", path, "\n")
    }

    safe_call <- function(fn) {
      tryCatch(fn(), error=function(e) e)
    }

    hash_df <- function(df) {
      if (!is.data.frame(df)) return("")
      dt <- data.table::as.data.table(df)
      data.table::setcolorder(dt, sort(names(dt)))
      if ("game_id" %in% names(dt)) data.table::setorder(dt, game_id)
      digest::digest(dt, algo="xxhash64")
    }

    should_change_check <- function(dataset, season) {
      if (is.null(season)) return(isTRUE(CHECK_STATIC_FOR_CHANGES))
      if (as.integer(season) == as.integer(CURRENT_SEASON)) return(isTRUE(CHECK_CURRENT_FOR_CHANGES))
      FALSE
    }

    call_with_year_or_season <- function(fn, yr, season_type) {
      fmls <- names(formals(fn))
      if ("season" %in% fmls) {
        fn(season = yr, season_type = season_type)
      } else if ("year" %in% fmls) {
        fn(year = yr, season_type = season_type)
      } else if ("season_end_year" %in% fmls) {
        fn(season_end_year = yr, season_type = season_type)
      } else {
        stop("Function does not accept season/year arguments in this hoopR version.")
      }
    }

    run_task <- function(dataset, season=NULL, season_type=NULL, path, fn) {
      file_exists <- file.exists(path)

      if (!file_exists) {
        cat("▶️ missing -> fetch:", dataset, season, season_type, "\n")
        res <- safe_call(fn)
        if (inherits(res, "error")) {
          msg <- conditionMessage(res)
          cat("⚠️ FAIL:", msg, "\n")
          log_row(dataset, season, season_type, path, "fail", msg, "")
          return(invisible(NULL))
        }
        if (is.null(res) || (is.data.frame(res) && nrow(res)==0)) {
          cat("⚠️ EMPTY\n")
          log_row(dataset, season, season_type, path, "fail", "empty result", "")
          return(invisible(NULL))
        }
        h <- clean_hash(hash_df(res))
        write_one(res, path)
        log_row(dataset, season, season_type, path, "ok", "missing -> wrote", h)
        return(invisible(NULL))
      }

      if (FORCE) {
        cat("♻️ FORCE overwrite:", dataset, season, season_type, "\n")
        res <- safe_call(fn)
        if (inherits(res, "error") || is.null(res) || (is.data.frame(res) && nrow(res)==0)) {
          msg <- if (inherits(res,"error")) conditionMessage(res) else "empty result"
          cat("⚠️ FAIL:", msg, "\n")
          log_row(dataset, season, season_type, path, "fail", msg, "")
          return(invisible(NULL))
        }
        h <- clean_hash(hash_df(res))
        write_one(res, path)
        log_row(dataset, season, season_type, path, "ok", "forced overwrite", h)
        return(invisible(NULL))
      }

      if (!should_change_check(dataset, season)) {
        cat("↩️ skip (exists):", dataset, season, season_type, "\n")
        log_row(dataset, season, season_type, path, "skip", "exists (no change-check)", "")
        return(invisible(NULL))
      }

      cat("🔍 check-changes:", dataset, season, season_type, "\n")
      res <- safe_call(fn)
      if (inherits(res, "error")) {
        msg <- conditionMessage(res)
        cat("⚠️ FAIL:", msg, "\n")
        log_row(dataset, season, season_type, path, "fail", msg, "")
        return(invisible(NULL))
      }
      if (is.null(res) || (is.data.frame(res) && nrow(res)==0)) {
        cat("⚠️ EMPTY\n")
        log_row(dataset, season, season_type, path, "fail", "empty result", "")
        return(invisible(NULL))
      }

      new_hash <- clean_hash(hash_df(res))
      old_hash <- clean_hash(last_ok_hash(dataset, season, season_type, path))

      if (old_hash == "" || new_hash != old_hash) {
        cat("♻️ changed -> overwrite:", dataset, season, season_type, "\n")
        write_one(res, path)
        log_row(dataset, season, season_type, path, "ok", "changed overwrite", new_hash)
      } else {
        cat("↩️ unchanged:", dataset, season, season_type, "\n")
        log_row(dataset, season, season_type, path, "skip", "unchanged", new_hash)
      }

      invisible(NULL)
    }

    if (EXPORT_TEAMS) {
      run_task("teams", NULL, NULL, out_path("teams/mbb_teams.parquet"),
               function() hoopR::espn_mbb_teams())
    }
    if (EXPORT_STANDINGS) {
      run_task("standings", NULL, NULL, out_path("standings/mbb_standings.parquet"),
               function() hoopR::espn_mbb_standings())
    }
    if (EXPORT_RANKINGS) {
      run_task("rankings", NULL, NULL, out_path("rankings/mbb_rankings.parquet"),
               function() hoopR::espn_mbb_rankings())
    }

    for (yr in SEASONS) {
      cat("\n=============================\n")
      cat("🏀 Season end year:", yr, "\n")
      cat("=============================\n")

      if (EXPORT_SCHEDULE) {
        run_task("schedule", yr, NULL, out_path(sprintf("schedule/mbb_schedule_%s.parquet", yr)),
                 function() hoopR::load_mbb_schedule(season=yr))
      }

      if (EXPORT_TEAM_BOX) {
        run_task("team_box", yr, NULL, out_path(sprintf("team_box/mbb_team_box_%s.parquet", yr)),
                 function() hoopR::load_mbb_team_box(season=yr))
      }

      if (EXPORT_PLAYER_BOX) {
        run_task("player_box", yr, NULL, out_path(sprintf("player_box/mbb_player_box_%s.parquet", yr)),
                 function() hoopR::load_mbb_player_box(season=yr))
      }

      if (EXPORT_TEAM_STATS) {
        for (stype in c("regular","postseason")) {
          run_task("team_stats", yr, stype, out_path(sprintf("team_stats/mbb_team_stats_%s_%s.parquet", yr, stype)),
                   function() call_with_year_or_season(hoopR::espn_mbb_team_stats, yr, stype))
        }
      }

      if (EXPORT_PLAYER_STATS) {
        for (stype in c("regular","postseason")) {
          run_task("player_stats", yr, stype, out_path(sprintf("player_stats/mbb_player_stats_%s_%s.parquet", yr, stype)),
                   function() call_with_year_or_season(hoopR::espn_mbb_player_stats, yr, stype))
        }
      }

      if (EXPORT_PBP) {
        run_task("pbp", yr, NULL, out_path(sprintf("pbp/mbb_pbp_%s.parquet", yr)),
                 function() hoopR::load_mbb_pbp(season=yr))
      }
    }

    cat("\n✅ DONE exporting hoopR → Parquet (missing-or-changed)\n")
    cat("🧾 Manifest:", manifest_path, "\n")
    '''

    r(r_code)


# ============================================================
# CELL 10B: DATA LOADERS + ROLLING SNAPSHOTS
# REQUIRED BY CELL 11 SCHEDULER
# ============================================================

def _read_any_table(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".parquet":
            return pd.read_parquet(path)
        if ext in [".csv", ".txt"]:
            return pd.read_csv(path, low_memory=False)
        if ext in [".xlsx", ".xls"]:
            return pd.read_excel(path)
    except Exception as e:
        print(f"⚠️ Failed reading {os.path.basename(path)}: {e}")
    return pd.DataFrame()


def _find_files_recursive(base_dir: str, exts=(".parquet", ".csv", ".xlsx", ".xls")) -> list:
    hits = []
    for root, _, files in os.walk(base_dir):
        for fn in files:
            if fn.lower().endswith(exts):
                hits.append(os.path.join(root, fn))
    return sorted(hits)


def _normalize_team_box_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Make team_box columns consistent enough for training/snapshots.
    """
    if df is None or len(df) == 0:
        return pd.DataFrame()

    x = df.copy()
    x.columns = [str(c).strip() for c in x.columns]

    # common aliases
    rename_map = {}

    colset = {c.lower(): c for c in x.columns}

    def pick(*names):
        for n in names:
            if n.lower() in colset:
                return colset[n.lower()]
        return None

    game_id_col = pick("game_id", "id")
    if game_id_col and game_id_col != "game_id":
        rename_map[game_id_col] = "game_id"

    season_col = pick("season", "season_year")
    if season_col and season_col != "season":
        rename_map[season_col] = "season"

    game_dt_col = pick("game_date_time", "start_date", "date", "game_date")
    if game_dt_col and game_dt_col != "game_date_time":
        rename_map[game_dt_col] = "game_date_time"

    team_id_col = pick("team_id", "team.id", "teamid")
    if team_id_col and team_id_col != "team_id":
        rename_map[team_id_col] = "team_id"

    opp_id_col = pick("opponent_team_id", "opponent_id", "opponent.id", "opponentteamid")
    if opp_id_col and opp_id_col != "opponent_team_id":
        rename_map[opp_id_col] = "opponent_team_id"

    team_score_col = pick("team_score", "score", "points")
    if team_score_col and team_score_col != "team_score":
        rename_map[team_score_col] = "team_score"

    opp_score_col = pick("opponent_team_score", "opp_score", "opponent_score", "points_against")
    if opp_score_col and opp_score_col != "opponent_team_score":
        rename_map[opp_score_col] = "opponent_team_score"

    team_name_col = pick("team_name", "team_display_name", "display_name", "team")
    if team_name_col and team_name_col != "team_name":
        rename_map[team_name_col] = "team_name"

    x = x.rename(columns=rename_map)

    # required core columns
    required = [
        "game_id", "season", "game_date_time",
        "team_id", "opponent_team_id",
        "team_score", "opponent_team_score"
    ]
    for c in required:
        if c not in x.columns:
            x[c] = np.nan

    # numeric coercions
    for c in ["team_id", "opponent_team_id", "team_score", "opponent_team_score", "season"]:
        x[c] = pd.to_numeric(x[c], errors="coerce")

    x["game_date_time"] = pd.to_datetime(x["game_date_time"], errors="coerce", utc=True)

    # add team_name if absent
    if "team_name" not in x.columns:
        x["team_name"] = np.nan

    x = x.dropna(subset=["game_id", "team_id", "opponent_team_id", "game_date_time"]).copy()

    x["game_id"] = x["game_id"].astype(str)
    x["team_id"] = x["team_id"].astype(int)
    x["opponent_team_id"] = x["opponent_team_id"].astype(int)

    return x.reset_index(drop=True)


def load_team_box_history(team_box_dir: str, seasons: list) -> pd.DataFrame:
    """
    Loads all team-box history files from TEAM_BOX_DIR recursively.
    Accepts parquet/csv/xlsx.
    Filters to requested seasons when season column exists.
    """
    files = _find_files_recursive(team_box_dir, exts=(".parquet", ".csv", ".xlsx", ".xls"))
    if not files:
        raise FileNotFoundError(f"No team_box files found in {team_box_dir}")

    dfs = []
    for f in files:
        df = _read_any_table(f)
        if len(df) == 0:
            continue
        df = _normalize_team_box_columns(df)
        if len(df) == 0:
            continue
        dfs.append(df)

    if not dfs:
        raise ValueError(f"No readable team_box data found in {team_box_dir}")

    out = pd.concat(dfs, ignore_index=True)

    if "season" in out.columns:
        out = out[out["season"].isin(seasons)].copy()

    out = out.sort_values(["game_date_time", "game_id", "team_id"]).reset_index(drop=True)
    out = out.drop_duplicates(subset=["game_id", "team_id"], keep="last").reset_index(drop=True)

    return out


def _normalize_schedule_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    x = df.copy()
    x.columns = [str(c).strip() for c in x.columns]
    colset = {c.lower(): c for c in x.columns}

    def pick(*names):
        for n in names:
            if n.lower() in colset:
                return colset[n.lower()]
        return None

    rename_map = {}

    for target, options in {
        "game_id": ["game_id", "id"],
        "season": ["season", "season_year"],
        "game_date_time": ["game_date_time", "start_date", "date", "game_date"],
        "home_id": ["home_id", "home_team_id", "home.id"],
        "away_id": ["away_id", "away_team_id", "away.id"],
        "home_team": ["home_team", "home_display_name", "home_name"],
        "away_team": ["away_team", "away_display_name", "away_name"],
        "home_short_display_name": ["home_short_display_name", "home_short_name"],
        "away_short_display_name": ["away_short_display_name", "away_short_name"],
        "home_score": ["home_score"],
        "away_score": ["away_score"],
        "neutral_site": ["neutral_site", "neutral"],
        "status_type_completed": ["status_type_completed"],
        "status_type_state": ["status_type_state"],
        "status_type_name": ["status_type_name"],
        "status_type_short_detail": ["status_type_short_detail"],
    }.items():
        src = pick(*options)
        if src and src != target:
            rename_map[src] = target

    x = x.rename(columns=rename_map)

    for c in [
        "game_id", "season", "game_date_time", "home_id", "away_id",
        "home_team", "away_team", "home_short_display_name", "away_short_display_name",
        "home_score", "away_score", "neutral_site",
        "status_type_completed", "status_type_state", "status_type_name", "status_type_short_detail"
    ]:
        if c not in x.columns:
            x[c] = np.nan

    x["game_date_time"] = pd.to_datetime(x["game_date_time"], errors="coerce", utc=True)
    for c in ["season", "home_id", "away_id", "home_score", "away_score"]:
        x[c] = pd.to_numeric(x[c], errors="coerce")

    x["game_id"] = x["game_id"].astype(str)
    x = x.dropna(subset=["game_id", "game_date_time"]).copy()

    return x.reset_index(drop=True)


def load_schedule(schedule_dir: str, season: int) -> pd.DataFrame:
    """
    Loads schedule files recursively and returns the requested season.
    """
    files = _find_files_recursive(schedule_dir, exts=(".parquet", ".csv", ".xlsx", ".xls"))
    if not files:
        raise FileNotFoundError(f"No schedule files found in {schedule_dir}")

    dfs = []
    for f in files:
        df = _read_any_table(f)
        if len(df) == 0:
            continue
        df = _normalize_schedule_columns(df)
        if len(df) == 0:
            continue
        dfs.append(df)

    if not dfs:
        raise ValueError(f"No readable schedule data found in {schedule_dir}")

    out = pd.concat(dfs, ignore_index=True)

    if "season" in out.columns and out["season"].notna().any():
        out = out[out["season"] == season].copy()

    out = out.sort_values(["game_date_time", "game_id"]).reset_index(drop=True)
    out = out.drop_duplicates(subset=["game_id"], keep="last").reset_index(drop=True)

    return out


def build_team_rolling_snapshots(team_box_all: pd.DataFrame, window: int = ROLL_WINDOW):
    """
    Builds one row per team per played game with leakage-safe rolling features.
    These snapshots are later merge_asof'd into upcoming slates.
    """
    tb = team_box_all.copy()
    if len(tb) == 0:
        return pd.DataFrame(), []

    tb["game_date_time"] = pd.to_datetime(tb["game_date_time"], errors="coerce", utc=True)
    tb["game_dt_et"] = tb["game_date_time"].dt.tz_convert("America/New_York").dt.tz_localize(None)

    # useful fallback stat creation
    if "points_for" not in tb.columns:
        tb["points_for"] = pd.to_numeric(tb["team_score"], errors="coerce")
    if "points_against" not in tb.columns:
        tb["points_against"] = pd.to_numeric(tb["opponent_team_score"], errors="coerce")

    if "point_diff" not in tb.columns:
        tb["point_diff"] = pd.to_numeric(tb["points_for"], errors="coerce") - pd.to_numeric(tb["points_against"], errors="coerce")

    # keep numeric non-ID columns
    exclude = {
        "game_id", "season", "game_date", "game_date_time", "game_dt_et",
        "team_id", "team_uid", "team_slug",
        "opponent_team_id", "opponent_team_uid", "opponent_team_slug",
    }

    numeric_cols = []
    for c in tb.columns:
        if c in exclude:
            continue
        if str(c).startswith("opponent_"):
            continue
        if pd.api.types.is_numeric_dtype(tb[c]):
            numeric_cols.append(c)

    # make sure these core stats exist
    for c in ["points_for", "points_against", "point_diff"]:
        if c not in numeric_cols and c in tb.columns:
            numeric_cols.append(c)

    tb = tb.sort_values(["team_id", "game_dt_et", "game_id"]).reset_index(drop=True)

    roll_cols = []
    for c in numeric_cols:
        tb[c] = pd.to_numeric(tb[c], errors="coerce")

        c1 = f"r{window}_mean_{c}"
        c2 = f"r3_mean_{c}"
        c3 = f"r{window}_std_{c}"

        tb[c1] = (
            tb.groupby("team_id")[c]
              .rolling(window, min_periods=3).mean()
              .shift(1)
              .reset_index(level=0, drop=True)
        )
        tb[c2] = (
            tb.groupby("team_id")[c]
              .rolling(3, min_periods=2).mean()
              .shift(1)
              .reset_index(level=0, drop=True)
        )
        tb[c3] = (
            tb.groupby("team_id")[c]
              .rolling(window, min_periods=3).std()
              .shift(1)
              .reset_index(level=0, drop=True)
        )

        roll_cols.extend([c1, c2, c3])

    tb[f"r{window}_gp"] = (
        tb.groupby("team_id")["team_id"]
          .rolling(window, min_periods=1).count()
          .shift(1)
          .reset_index(level=0, drop=True)
    )
    roll_cols.append(f"r{window}_gp")

    # rest days
    tb["prev_game_dt_et"] = tb.groupby("team_id")["game_dt_et"].shift(1)
    tb["rest_days"] = (tb["game_dt_et"] - tb["prev_game_dt_et"]).dt.total_seconds() / 86400.0
    tb["rest_days"] = tb["rest_days"].clip(lower=0)
    roll_cols.append("rest_days")

    keep_cols = ["team_id", "game_id", "game_dt_et"]
    if "team_name" in tb.columns:
        keep_cols.append("team_name")

    out = tb[keep_cols + [c for c in roll_cols if c in tb.columns]].copy()
    out = out.sort_values(["game_dt_et", "team_id"]).reset_index(drop=True)

    return out, [c for c in roll_cols if c in out.columns]


# ============================================================
# CELL 11: AUTO-RETRAINING SCHEDULER
# FULL REWRITE — matches CELL 5 + CELL 8 + CELL 9
# ============================================================

def check_and_retrain(force_data: bool = False, force_model: bool = False):
    """
    Checks retraining timestamps and retrains as needed.
    - Data refresh: every RETRAIN_DATA_HOURS
    - Model refit: every RETRAIN_MODEL_DAYS
    """
    global team_snaps, roll_cols, spread_booster, winner_booster, lgb_spread, iso, imp
    global diff_cols, diff_cols2, feature_cols, dataset_all, schedule_cur, elo_snap, cur2
    global team_box_hist

    meta = _load_metadata()
    now_str = datetime.utcnow().isoformat()

    need_data = force_data or _hours_since(meta.get("last_data_refresh")) >= RETRAIN_DATA_HOURS
    need_model = force_model or _days_since(meta.get("last_model_fit")) >= RETRAIN_MODEL_DAYS

    if need_data:
        print(f"⏳ Data refresh triggered (last: {meta.get('last_data_refresh', 'never')})")

        run_hoopr_refresh(
            raw_dir=RAW_DIR,
            current_season=CURRENT_SEASON,
            seasons=HIST_SEASONS,
            force=False,
            check_changes_for_current=True,
            export={
                "schedule": True,
                "team_box": True,
                "player_box": True,
                "pbp": True,
                "teams": True,
                "standings": True,
                "rankings": True,
                "team_stats": True,
                "player_stats": True,
            },
        )

        meta["last_data_refresh"] = now_str
        _save_metadata(meta)

    print("📂 Loading team box history...")
    team_box_hist = load_team_box_history(TEAM_BOX_DIR, HIST_SEASONS)
    tb_hist = team_box_hist.copy()
    schedule_cur = load_schedule(SCHEDULE_DIR, CURRENT_SEASON)
    print(f"  team_box rows: {len(tb_hist):,}  |  schedule rows: {len(schedule_cur):,}")

    print("📊 Building rolling snapshots + Elo...")
    team_snaps, roll_cols = build_team_rolling_snapshots(tb_hist, window=ROLL_WINDOW)

    # full Elo history for as-of merges
    elo_snap = compute_elo_ratings(tb_hist)

    if need_model:
        print(f"🔁 Full model retrain triggered (last: {meta.get('last_model_fit', 'never')})")
        print("📐 Building training dataset...")

        dataset_all = build_game_dataset_from_team_box(
            team_box=tb_hist,
            schedule_df=schedule_cur,
            window=ROLL_WINDOW,
            elo_df=elo_snap,
        )
        dataset_all = dataset_all.sort_values("game_dt_et").reset_index(drop=True)
        print(f"  Training rows: {len(dataset_all):,}")

        spread_booster, winner_booster, lgb_spread, iso, imp, diff_cols2, feature_cols = \
            train_models(dataset_all, CURRENT_SEASON)

        diff_cols = [c for c in diff_cols2 if not c.endswith("__na")]

        save_models(spread_booster, winner_booster, lgb_spread, iso, imp, MODEL_DIR)
        meta["last_model_fit"] = now_str
        meta["feature_cols"] = feature_cols
        _save_metadata(meta)
        print("✅ Model retrain complete")

    else:
        print(f"⏭️  Model is fresh (last fit: {meta.get('last_model_fit', 'never')}). Loading from disk.")
        try:
            spread_booster, winner_booster, lgb_spread, iso, imp = load_models(MODEL_DIR)
            meta_cols = meta.get("feature_cols", [])
            diff_cols2 = meta_cols
            diff_cols = [c for c in meta_cols if not c.endswith("__na")]
            feature_cols = meta_cols
        except Exception as e:
            print(f"⚠️ Could not load saved models ({e}). Triggering fresh model train...")
            return check_and_retrain(force_data=False, force_model=True)

    cur2 = None
    print("✅ Scheduler complete. All globals ready.")


print("🚀 Running startup retrain check...")
check_and_retrain()
#check_and_retrain(force_data=True, force_model=False)


# ============================================================
# CELL 12: PREDICTION ENGINE
# ============================================================

def make_xgb_matrix(pregame: pd.DataFrame, booster, imp_ref=None) -> xgb.DMatrix:
    """Builds an XGBoost DMatrix matching the booster's feature names."""
    fnames = list(booster.feature_names)
    base_diff = [c for c in fnames if c.startswith("diff_") and not c.endswith("__na")]
    na_cols   = [c for c in fnames if c.endswith("__na")]

    X = pregame.copy()
    for c in base_diff:
        if c not in X.columns:
            X[c] = np.nan

    if na_cols:
        flags = pd.DataFrame(index=X.index)
        for nc in na_cols:
            bc = nc[:-4]
            if nc not in X.columns:
                flags[nc] = X[bc].isna().astype("int8")
        X = pd.concat([X, flags], axis=1)

    if imp_ref is not None and base_diff:
        X[base_diff] = imp_ref.transform(X[base_diff])

    X = X.reindex(columns=fnames, fill_value=0.0)
    if X.columns.duplicated().any():
        X = X.loc[:, ~X.columns.duplicated()]
    return xgb.DMatrix(X, feature_names=fnames)


def predict_slate(
    pregame: pd.DataFrame,
    spread_booster, winner_booster, lgb_spread,
    iso, imp, feature_cols,
    ensemble_weight_lgb: float = 0.25,    # blend weight for LGB margin
) -> pd.DataFrame:
    """
    Runs ensemble prediction on a pregame DataFrame.
    Returns pregame with columns added:
      pred_margin_home, p_home_win, winner_pick, pick_conf, spread_pick
    """
    if pregame is None or len(pregame) == 0:
        return pd.DataFrame()

    dcur_spread = make_xgb_matrix(pregame, spread_booster, imp)
    dcur_winner = make_xgb_matrix(pregame, winner_booster, imp)

    xgb_margin = spread_booster.predict(dcur_spread, iteration_range=(0, spread_booster.best_iteration + 1))

    # LGB margin using same feature set
    fnames = list(spread_booster.feature_names)
    X_lgb = pregame.reindex(columns=fnames, fill_value=0.0)
    lgb_margin = lgb_spread.predict(X_lgb.values)

    # Ensemble margin: weighted blend
    pred_margin_ab = (1 - ensemble_weight_lgb) * xgb_margin + ensemble_weight_lgb * lgb_margin

    raw_pA = winner_booster.predict(dcur_winner, iteration_range=(0, winner_booster.best_iteration + 1))
    pred_pA = iso.transform(raw_pA) if iso is not None else raw_pA

    # Home/Away conversion
    teamA_is_home = (pregame["teamA_id"].astype(int) == pregame["home_id"].astype(int)).values
    pred_margin_home = np.where(teamA_is_home, pred_margin_ab, -pred_margin_ab).astype(float)
    p_home_win_raw   = np.where(teamA_is_home, pred_pA, 1 - pred_pA).astype(float)

    # Clip for display (never show 0% / 100%)
    EPS = 0.02
    p_home_win = np.clip(p_home_win_raw, EPS, 1 - EPS)

    pred_margin_home = np.round(pred_margin_home, 1)
    pred_margin_home[np.abs(pred_margin_home) < 0.05] = 0.0

    winner_pick = np.where(p_home_win_raw >= 0.5, pregame["home_team"].values, pregame["away_team"].values)
    pick_conf   = np.where(p_home_win >= 0.5, p_home_win, 1 - p_home_win)
    spread_pick = np.where(pred_margin_home >= 0, pregame["home_team"].values, pregame["away_team"].values)

    model_line = np.where(
        pred_margin_home >= 0,
        pregame["home_team"].values + " -" + np.abs(pred_margin_home).round(1).astype(str),
        pregame["away_team"].values + " -" + np.abs(pred_margin_home).round(1).astype(str),
    )

    return pregame.assign(
        pred_margin_home=pred_margin_home,
        p_home_win=p_home_win,
        p_home_win_raw=p_home_win_raw,
        winner_pick=winner_pick,
        pick_conf=pick_conf,
        spread_pick=spread_pick,
        model_line_text=model_line,
    ).copy()


# ============================================================
# CELL 13: BOARD BUILDER
# ============================================================

def build_board_for_date(
    date_et,
    schedule_df, team_snaps, roll_cols,
    spread_booster, winner_booster, lgb_spread,
    iso, imp, feature_cols,
    elo_snap=None,
    verbose=False,
) -> pd.DataFrame:
    """Full pipeline: pregame features → predictions → actual score backfill → injury merge."""
    with (suppress_stdout_stderr() if not verbose else contextlib.nullcontext()):
        pregame = build_pregame_dataset_for_slate(
            schedule_df=schedule_df,
            team_snaps=team_snaps,
            roll_cols=roll_cols,
            slate_date_et=date_et,
            window=ROLL_WINDOW,
            elo_snap=elo_snap,
        )

    if pregame is None or len(pregame) == 0:
        return pd.DataFrame()

    pregame = normalize_board_for_downstream(pregame)

    board = predict_slate(
        pregame, spread_booster, winner_booster, lgb_spread,
        iso, imp, feature_cols
    )

    board = normalize_board_for_downstream(board)

    # first try schedule backfill
    board = attach_actual_scores_from_schedule(board, schedule_df, date_et)
    board = normalize_board_for_downstream(board)

    # then fill any remaining missing finals from team_box history
    if "team_box_hist" in globals() and team_box_hist is not None and len(team_box_hist) > 0:
        board = attach_actual_scores_from_team_box(board, team_box_hist, date_et)
        board = normalize_board_for_downstream(board)

    board = attach_fresh_scores_from_hoopr(board, date_et)

    # Attach final scores for completed games
    board["home_final"] = pd.to_numeric(board.get("home_score"), errors="coerce")
    board["away_final"] = pd.to_numeric(board.get("away_score"), errors="coerce")

    completed_bool = pd.Series(False, index=board.index)
    for col, vals in [
        ("status_type_completed", ["true", "1", "yes"]),
        ("status_type_state", ["post", "postgame", "final"]),
    ]:
        if col in board.columns:
            completed_bool |= board[col].astype(str).str.lower().isin(vals)

    if "status_type_short_detail" in board.columns:
        completed_bool |= board["status_type_short_detail"].astype(str).str.contains("Final", case=False, na=False)

    finished = completed_bool & board["home_final"].notna() & board["away_final"].notna()

    board["final_score"] = ""
    if finished.any():
        away_p = board.loc[finished, "away_final"].round(0).astype(int).astype(str)
        home_p = board.loc[finished, "home_final"].round(0).astype(int).astype(str)
        board.loc[finished, "final_score"] = (
            board.loc[finished, "away_team"] + " " + away_p + " - " +
            board.loc[finished, "home_team"] + " " + home_p
        )

    # Attach injury features
    board = attach_injury_features_to_board(board, date_et, INJURY_DIR)
    board = normalize_board_for_downstream(board)

    # dedupe
    dedupe_cols = [c for c in ["game_id", "away_team", "home_team", "game_dt_et"] if c in board.columns]
    if dedupe_cols:
        board = board.drop_duplicates(subset=dedupe_cols, keep="last").reset_index(drop=True)

    return board


def _attach_rotowire_to_board(board: pd.DataFrame, slate_date_et) -> pd.DataFrame:
    """Attaches Rotowire odds to board with home/away flip protection."""
    outb = board.copy()
    rw_cols = ["rw_spread_home", "rw_home_ml", "rw_away_ml", "rw_total", "rw_missing_reason"]
    outb = outb.drop(columns=[c for c in rw_cols if c in outb.columns], errors="ignore")

    mkt = load_rotowire_all_for_date(slate_date_et, ROTOWIRE_DIR)
    if mkt is None or len(mkt) == 0:
        outb["rw_missing_reason"] = "No RW odds in file"
        return outb

    b = outb.copy()
    m = mkt.copy()
    b["k_away"] = b["away_team"].map(canonical_team)
    b["k_home"] = b["home_team"].map(canonical_team)
    m["k_away"] = m["away_team"].map(canonical_team)
    m["k_home"] = m["home_team"].map(canonical_team)
    b["k_game"] = b.apply(lambda r: "||".join(sorted([r["k_away"], r["k_home"]])), axis=1)
    m["k_game"] = m.apply(lambda r: "||".join(sorted([r["k_away"], r["k_home"]])), axis=1)
    m = m.drop_duplicates("k_game", keep="last")

    merged = b.merge(
        m[["k_game","rw_spread_home","rw_home_ml","rw_away_ml","rw_total","k_home","k_away"]],
        on="k_game", how="left", suffixes=("", "_rw")
    )

    if "k_home_rw" in merged.columns:
        same_home = merged["home_team"].map(canonical_team) == merged["k_home_rw"]
        merged["rw_spread_home"] = np.where(same_home, merged["rw_spread_home"], -merged["rw_spread_home"])
        old_hml = merged["rw_home_ml"].copy()
        old_aml = merged["rw_away_ml"].copy()
        merged["rw_home_ml"] = np.where(same_home, old_hml, old_aml)
        merged["rw_away_ml"] = np.where(same_home, old_aml, old_hml)

    merged = merged.drop(columns=["k_away","k_home","k_game","k_home_rw","k_away_rw"], errors="ignore")
    has_rw = merged[["rw_spread_home","rw_home_ml","rw_away_ml","rw_total"]].notna().any(axis=1)
    merged["rw_missing_reason"] = np.where(has_rw, "", "No RW odds in file")
    return merged


# ============================================================
# CELL 13B: BACKFILL ACTUAL FINAL SCORES FROM SCHEDULE
# ============================================================

def attach_actual_scores_from_schedule(board: pd.DataFrame, schedule_df: pd.DataFrame, slate_date_et) -> pd.DataFrame:
    """
    Backfill completed scores/status from schedule onto a board built from
    schedule rows or Rotowire fallback rows.

    Matching priority:
      1) exact game_id
      2) same-date exact oriented canonical away/home teams
      3) same-date unordered canonical matchup, with score flip if needed
    """
    if board is None or len(board) == 0:
        return pd.DataFrame()

    out = normalize_board_for_downstream(board.copy())
    if schedule_df is None or len(schedule_df) == 0:
        return out

    sched = schedule_df.copy()

    # -----------------------------
    # normalize schedule datetime
    # -----------------------------
    dt_col = None
    for c in ["game_dt_et", "game_date_time", "start_date", "date"]:
        if c in sched.columns:
            dt_col = c
            break
    if dt_col is None:
        return out

    sched[dt_col] = pd.to_datetime(sched[dt_col], errors="coerce", utc=True)
    if pd.api.types.is_datetime64tz_dtype(sched[dt_col]):
        sched["game_dt_et"] = sched[dt_col].dt.tz_convert("America/New_York").dt.tz_localize(None)
    else:
        sched["game_dt_et"] = pd.to_datetime(sched[dt_col], errors="coerce")

    sched = sched.dropna(subset=["game_dt_et"]).copy()
    sched["slate_date_et"] = sched["game_dt_et"].dt.date
    sched = sched[sched["slate_date_et"] == pd.Timestamp(slate_date_et).date()].copy()

    if len(sched) == 0:
        return out

    # -----------------------------
    # standardize schedule columns
    # -----------------------------
    if "game_id" in sched.columns:
        sched["game_id"] = sched["game_id"].astype(str)

    for c in ["home_score", "away_score", "home_id", "away_id"]:
        if c in sched.columns:
            sched[c] = pd.to_numeric(sched[c], errors="coerce")

    if "home_team" not in sched.columns:
        if "home_short_display_name" in sched.columns:
            sched["home_team"] = sched["home_short_display_name"].astype(str)
        else:
            sched["home_team"] = "TBD"

    if "away_team" not in sched.columns:
        if "away_short_display_name" in sched.columns:
            sched["away_team"] = sched["away_short_display_name"].astype(str)
        else:
            sched["away_team"] = "TBD"

    sched["home_team"] = sched["home_team"].astype(str)
    sched["away_team"] = sched["away_team"].astype(str)

    out["home_team"] = out["home_team"].astype(str)
    out["away_team"] = out["away_team"].astype(str)

    # canonical keys
    sched["k_home"] = sched["home_team"].map(canonical_team)
    sched["k_away"] = sched["away_team"].map(canonical_team)
    sched["k_game"] = sched.apply(lambda r: "||".join(sorted([r["k_away"], r["k_home"]])), axis=1)

    out["k_home"] = out["home_team"].map(canonical_team)
    out["k_away"] = out["away_team"].map(canonical_team)
    out["k_game"] = out.apply(lambda r: "||".join(sorted([r["k_away"], r["k_home"]])), axis=1)
    out["slate_date_et"] = pd.to_datetime(out["game_dt_et"], errors="coerce").dt.date

    # prefer rows that actually have finals
    sched["has_scores"] = sched["home_score"].notna() & sched["away_score"].notna()
    sched = sched.sort_values(["has_scores", "game_dt_et"], ascending=[False, True]).copy()

    # preserve original board scores/status so we can coalesce cleanly
    for c in ["home_score", "away_score", "status_type_completed", "status_type_state", "status_type_name", "status_type_short_detail"]:
        if c not in out.columns:
            out[c] = np.nan

    # -----------------------------
    # 1) exact game_id match
    # -----------------------------
    if "game_id" in out.columns and "game_id" in sched.columns:
        gid_lookup = (
            sched.drop_duplicates(subset=["game_id"], keep="first")
                 [["game_id", "home_score", "away_score",
                   "status_type_completed", "status_type_state",
                   "status_type_name", "status_type_short_detail"]]
                 .copy()
        )

        out = out.merge(gid_lookup, on="game_id", how="left", suffixes=("", "_gid"))

        for c in ["home_score", "away_score", "status_type_completed", "status_type_state", "status_type_name", "status_type_short_detail"]:
            cg = f"{c}_gid"
            if cg in out.columns:
                out[c] = out[c].where(out[c].notna(), out[cg])

        out = out.drop(columns=[c for c in out.columns if c.endswith("_gid")], errors="ignore")

    # -----------------------------
    # 2) exact oriented team match
    # -----------------------------
    still_missing = out["home_score"].isna() | out["away_score"].isna()
    if still_missing.any():
        orient_lookup = (
            sched.drop_duplicates(subset=["slate_date_et", "k_away", "k_home"], keep="first")
                 [["slate_date_et", "k_away", "k_home",
                   "home_score", "away_score",
                   "status_type_completed", "status_type_state",
                   "status_type_name", "status_type_short_detail"]]
                 .copy()
        )

        tmp = out.loc[still_missing].merge(
            orient_lookup,
            on=["slate_date_et", "k_away", "k_home"],
            how="left",
            suffixes=("", "_orient")
        )

        for c in ["home_score", "away_score", "status_type_completed", "status_type_state", "status_type_name", "status_type_short_detail"]:
            co = f"{c}_orient"
            if co in tmp.columns:
                tmp[c] = tmp[c].where(tmp[c].notna(), tmp[co])

        out.loc[still_missing, ["home_score", "away_score", "status_type_completed",
                                "status_type_state", "status_type_name", "status_type_short_detail"]] = \
            tmp[["home_score", "away_score", "status_type_completed",
                 "status_type_state", "status_type_name", "status_type_short_detail"]].values

    # -----------------------------
    # 3) unordered matchup fallback
    # -----------------------------
    still_missing = out["home_score"].isna() | out["away_score"].isna()
    if still_missing.any():
        unordered_lookup = (
            sched.drop_duplicates(subset=["slate_date_et", "k_game"], keep="first")
                 [["slate_date_et", "k_game", "k_home", "k_away",
                   "home_score", "away_score",
                   "status_type_completed", "status_type_state",
                   "status_type_name", "status_type_short_detail"]]
                 .copy()
        )

        tmp = out.loc[still_missing].merge(
            unordered_lookup,
            on=["slate_date_et", "k_game"],
            how="left",
            suffixes=("", "_unord")
        )

        same_home = tmp["k_home"] == tmp["k_home_unord"]

        # if schedule home matches board home, keep scores as-is; otherwise flip them
        tmp["home_score_fix"] = np.where(same_home, tmp["home_score_unord"], tmp["away_score_unord"])
        tmp["away_score_fix"] = np.where(same_home, tmp["away_score_unord"], tmp["home_score_unord"])

        out.loc[still_missing, "home_score"] = out.loc[still_missing, "home_score"].where(
            out.loc[still_missing, "home_score"].notna(), tmp["home_score_fix"].values
        )
        out.loc[still_missing, "away_score"] = out.loc[still_missing, "away_score"].where(
            out.loc[still_missing, "away_score"].notna(), tmp["away_score_fix"].values
        )

        for c in ["status_type_completed", "status_type_state", "status_type_name", "status_type_short_detail"]:
            cu = f"{c}_unord"
            if cu in tmp.columns:
                out.loc[still_missing, c] = out.loc[still_missing, c].where(
                    out.loc[still_missing, c].notna(), tmp[cu].values
                )

    out = out.drop(columns=["k_home", "k_away", "k_game", "slate_date_et"], errors="ignore")
    return out


# ============================================================
# CELL 13C: BACKFILL ACTUAL SCORES FROM TEAM BOX HISTORY
# ============================================================

def attach_actual_scores_from_team_box(board: pd.DataFrame, team_box_df: pd.DataFrame, slate_date_et) -> pd.DataFrame:
    """
    Backfill finals from team_box history using TEAM IDS and nearest-date matching.
    This avoids exact-date misses caused by timezone/date parsing differences.
    """
    if board is None or len(board) == 0:
        return pd.DataFrame()
    if team_box_df is None or len(team_box_df) == 0:
        return board.copy()

    out = normalize_board_for_downstream(board.copy())
    target_date = pd.Timestamp(slate_date_et).date()

    tb = team_box_df.copy()
    tb["game_id"] = tb["game_id"].astype(str)
    tb["game_date_time"] = pd.to_datetime(tb["game_date_time"], errors="coerce", utc=True)
    tb = tb.dropna(subset=["game_id", "game_date_time"]).copy()

    tb["game_dt_et"] = tb["game_date_time"].dt.tz_convert("America/New_York").dt.tz_localize(None)
    tb["game_date_et"] = tb["game_dt_et"].dt.date

    tb["team_id"] = pd.to_numeric(tb["team_id"], errors="coerce")
    tb["opponent_team_id"] = pd.to_numeric(tb["opponent_team_id"], errors="coerce")
    tb["team_score"] = pd.to_numeric(tb["team_score"], errors="coerce")
    tb["opponent_team_score"] = pd.to_numeric(tb["opponent_team_score"], errors="coerce")

    tb = tb.dropna(subset=[
        "team_id", "opponent_team_id", "team_score", "opponent_team_score"
    ]).copy()

    tb["team_id"] = tb["team_id"].astype("int64")
    tb["opponent_team_id"] = tb["opponent_team_id"].astype("int64")

    # unordered id pair
    tb["id_low"] = np.minimum(tb["team_id"], tb["opponent_team_id"]).astype("int64")
    tb["id_high"] = np.maximum(tb["team_id"], tb["opponent_team_id"]).astype("int64")

    # keep one row per game/team
    tb = tb.sort_values(["game_dt_et", "game_id", "team_id"]).drop_duplicates(
        subset=["game_id", "team_id"], keep="last"
    )

    # two team rows -> one game row
    a = tb[["game_id", "game_dt_et", "game_date_et", "id_low", "id_high", "team_id", "team_score"]].copy()
    b = tb[["game_id", "team_id", "team_score"]].copy()

    a = a.rename(columns={"team_id": "team1_id", "team_score": "team1_score"})
    b = b.rename(columns={"team_id": "team2_id", "team_score": "team2_score"})

    games = a.merge(b, on="game_id", how="inner")
    games = games[games["team1_id"] != games["team2_id"]].copy()

    # one row per actual game
    games = games.drop_duplicates(subset=["game_id", "id_low", "id_high"], keep="last").copy()

    # keep only games reasonably close to target date
    games["date_diff_days"] = (pd.to_datetime(games["game_date_et"]) - pd.Timestamp(target_date)).dt.days.abs()
    games = games[games["date_diff_days"] <= 2].copy()

    if len(games) == 0:
        return out

    # normalize board ids
    out["home_id"] = pd.to_numeric(out["home_id"], errors="coerce")
    out["away_id"] = pd.to_numeric(out["away_id"], errors="coerce")
    out = out.dropna(subset=["home_id", "away_id"]).copy()

    out["home_id"] = out["home_id"].astype("int64")
    out["away_id"] = out["away_id"].astype("int64")
    out["id_low"] = np.minimum(out["home_id"], out["away_id"]).astype("int64")
    out["id_high"] = np.maximum(out["home_id"], out["away_id"]).astype("int64")

    # for each board row, choose nearest matching completed game
    filled_home = []
    filled_away = []
    filled_done = []

    for _, r in out.iterrows():
        cand = games[(games["id_low"] == r["id_low"]) & (games["id_high"] == r["id_high"])].copy()

        if len(cand) == 0:
            filled_home.append(np.nan)
            filled_away.append(np.nan)
            filled_done.append(False)
            continue

        cand = cand.sort_values(["date_diff_days", "game_dt_et", "game_id"]).copy()
        g = cand.iloc[0]

        home_is_team1 = int(r["home_id"]) == int(g["team1_id"])
        if home_is_team1:
            hs = g["team1_score"]
            aw = g["team2_score"]
        else:
            hs = g["team2_score"]
            aw = g["team1_score"]

        filled_home.append(hs)
        filled_away.append(aw)
        filled_done.append(pd.notna(hs) and pd.notna(aw))

    miss = out["home_score"].isna() | out["away_score"].isna()

    out.loc[miss, "home_score"] = out.loc[miss, "home_score"].where(
        out.loc[miss, "home_score"].notna(), pd.Series(filled_home, index=out.index)[miss]
    )
    out.loc[miss, "away_score"] = out.loc[miss, "away_score"].where(
        out.loc[miss, "away_score"].notna(), pd.Series(filled_away, index=out.index)[miss]
    )

    done_mask = pd.Series(filled_done, index=out.index) & miss

    if "status_type_completed" not in out.columns:
        out["status_type_completed"] = False
    if "status_type_state" not in out.columns:
        out["status_type_state"] = ""
    if "status_type_short_detail" not in out.columns:
        out["status_type_short_detail"] = ""

    out.loc[done_mask, "status_type_completed"] = True
    out.loc[done_mask, "status_type_state"] = "post"
    out.loc[done_mask, "status_type_short_detail"] = "Final"

    out = out.drop(columns=["id_low", "id_high"], errors="ignore")
    return out

# ============================================================
# CELL 13D: YESTERDAY / SAME-DAY FINAL SCORE FALLBACK VIA hoopR
# Uses hoopR game-level ESPN summary endpoints for fresher finals
# PATCHED:
#   - suppresses noisy R console warnings
#   - removes duplicate function definitions
#   - dedupes scoreboard / fresh games cleanly
# ============================================================

from rpy2.robjects import r, pandas2ri
from rpy2.robjects.conversion import localconverter
import rpy2.rinterface_lib.callbacks as rcb
from contextlib import contextmanager


@contextmanager
def suppress_r_console():
    """
    Temporarily suppress R console output forwarded through rpy2.
    """
    old_print = rcb.consolewrite_print
    old_warn = rcb.consolewrite_warnerror
    rcb.consolewrite_print = lambda x: None
    rcb.consolewrite_warnerror = lambda x: None
    try:
        yield
    finally:
        rcb.consolewrite_print = old_print
        rcb.consolewrite_warnerror = old_warn


def fetch_mbb_scoreboard_game_ids_r(date_et) -> pd.DataFrame:
    """
    Returns one row per game on a specific date from hoopR::espn_mbb_scoreboard("YYYYMMDD").
    Dedupe by game_id because hoopR can return duplicate rows on some dates.
    """
    date_str = pd.Timestamp(date_et).strftime("%Y%m%d")
    r.assign("DATE_STR", date_str)

    r_code = r'''
    suppressPackageStartupMessages(library(hoopR))

    sb <- tryCatch(
      hoopR::espn_mbb_scoreboard(DATE_STR),
      error = function(e) data.frame()
    )

    if (!is.data.frame(sb) || nrow(sb) == 0) {
      out <- data.frame()
    } else {
      keep <- intersect(
        c("game_id",
          "home_team_id", "away_team_id",
          "home_team_display_name", "away_team_display_name",
          "home_team_short_display_name", "away_team_short_display_name",
          "game_date_time"),
        names(sb)
      )
      out <- sb[, keep, drop = FALSE]

      if ("game_id" %in% names(out)) {
        out$game_id <- as.character(out$game_id)
        out <- out[!duplicated(out$game_id), , drop = FALSE]
      }
    }

    out
    '''

    with suppress_r_console():
        with localconverter(pandas2ri.converter):
            df = r(r_code)

    if df is None or len(df) == 0:
        return pd.DataFrame()

    out = pd.DataFrame(df)
    if "game_id" in out.columns:
        out["game_id"] = out["game_id"].astype(str)
    return out


def fetch_mbb_team_boxes_for_game_ids_r(game_ids) -> pd.DataFrame:
    """
    Pull fresh game-level team box scores using hoopR::espn_mbb_team_box(game_id).
    Suppresses noisy console messages for unavailable IDs.
    """
    game_ids = [str(g) for g in game_ids if pd.notna(g)]
    if len(game_ids) == 0:
        return pd.DataFrame()

    r.assign("GAME_IDS", game_ids)

    r_code = r'''
    suppressPackageStartupMessages(library(hoopR))
    suppressPackageStartupMessages(library(dplyr))

    pull_one <- function(gid) {
      tryCatch({
        zz <- capture.output({
          x <- suppressMessages(
                 suppressWarnings(
                   hoopR::espn_mbb_team_box(as.numeric(gid))
                 )
               )
        })
        if (!is.data.frame(x) || nrow(x) == 0) return(data.frame())
        x$game_id <- as.character(x$game_id)
        x
      }, error = function(e) {
        data.frame()
      })
    }

    lst <- lapply(GAME_IDS, pull_one)
    lst <- lst[vapply(lst, nrow, integer(1)) > 0]

    if (length(lst) == 0) {
      out <- data.frame()
    } else {
      out <- bind_rows(lst)
    }

    out
    '''

    with suppress_r_console():
        with localconverter(pandas2ri.converter):
            df = r(r_code)

    if df is None or len(df) == 0:
        return pd.DataFrame()

    out = pd.DataFrame(df)
    if "game_id" in out.columns:
        out["game_id"] = out["game_id"].astype(str)

    # hard dedupe
    dedupe_cols = [c for c in ["game_id", "team_id"] if c in out.columns]
    if dedupe_cols:
        out = out.drop_duplicates(subset=dedupe_cols, keep="last").reset_index(drop=True)

    return out


def build_fresh_team_box_scores_for_date(date_et) -> pd.DataFrame:
    """
    Returns one row per game with fresh home/away final scores for a date.
    Uses scoreboard -> game_ids -> espn_mbb_team_box(game_id).
    """
    sb = fetch_mbb_scoreboard_game_ids_r(date_et)
    if sb is None or len(sb) == 0 or "game_id" not in sb.columns:
        return pd.DataFrame()

    team_boxes = fetch_mbb_team_boxes_for_game_ids_r(sb["game_id"].tolist())
    if team_boxes is None or len(team_boxes) == 0:
        return pd.DataFrame()

    tb = team_boxes.copy()

    for c in ["game_id", "team_id", "opponent_team_id", "team_score", "opponent_team_score"]:
        if c not in tb.columns:
            tb[c] = np.nan

    tb["game_id"] = tb["game_id"].astype(str)
    tb["team_id"] = pd.to_numeric(tb["team_id"], errors="coerce")
    tb["opponent_team_id"] = pd.to_numeric(tb["opponent_team_id"], errors="coerce")
    tb["team_score"] = pd.to_numeric(tb["team_score"], errors="coerce")
    tb["opponent_team_score"] = pd.to_numeric(tb["opponent_team_score"], errors="coerce")

    # --------------------------------------------------------
    # preferred path: explicit home/away
    # --------------------------------------------------------
    if "team_home_away" in tb.columns:
        home = tb[tb["team_home_away"].astype(str).str.lower().eq("home")].copy()
        away = tb[tb["team_home_away"].astype(str).str.lower().eq("away")].copy()

        if len(home) and len(away):
            home = home[["game_id", "team_id", "team_score"]].rename(
                columns={"team_id": "home_id", "team_score": "home_score"}
            )
            away = away[["game_id", "team_id", "team_score"]].rename(
                columns={"team_id": "away_id", "team_score": "away_score"}
            )

            out = home.merge(away, on="game_id", how="inner")
            out["status_type_completed"] = True
            out["status_type_state"] = "post"
            out["status_type_short_detail"] = "Final"

            out = out.drop_duplicates(subset=["game_id"], keep="last").reset_index(drop=True)
            return out

    # --------------------------------------------------------
    # fallback path: infer from two team rows
    # --------------------------------------------------------
    tb = tb.dropna(subset=["game_id", "team_id", "opponent_team_id", "team_score"]).copy()
    tb = tb.drop_duplicates(subset=["game_id", "team_id"], keep="last").reset_index(drop=True)

    rows = []
    for gid, g in tb.groupby("game_id", sort=False):
        g = g.drop_duplicates(subset=["team_id"], keep="last").copy()
        if len(g) != 2:
            continue

        r1 = g.iloc[0]
        r2 = g.iloc[1]

        rows.append({
            "game_id": str(gid),
            "team1_id": int(r1["team_id"]),
            "team2_id": int(r2["team_id"]),
            "team1_score": float(r1["team_score"]),
            "team2_score": float(r2["team_score"]),
            "status_type_completed": True,
            "status_type_state": "post",
            "status_type_short_detail": "Final",
        })

    out = pd.DataFrame(rows)
    if len(out):
        out = out.drop_duplicates(subset=["game_id"], keep="last").reset_index(drop=True)
    return out


def attach_fresh_scores_from_hoopr(board: pd.DataFrame, date_et) -> pd.DataFrame:
    """
    Merge freshest available finals from scoreboard/team_box-by-game onto the board.
    Only queries scoreboard games that match board team IDs.
    Also overwrites stale placeholder rows like 0-0 / pre.
    """
    if board is None or len(board) == 0:
        return pd.DataFrame()

    out = board.copy()

    for c in ["home_score", "away_score"]:
        if c not in out.columns:
            out[c] = np.nan
    for c in ["status_type_completed", "status_type_state", "status_type_short_detail"]:
        if c not in out.columns:
            out[c] = np.nan

    # --------------------------------------------------------
    # pull scoreboard rows for the date
    # --------------------------------------------------------
    sb = fetch_mbb_scoreboard_game_ids_r(date_et)
    if sb is None or len(sb) == 0 or "game_id" not in sb.columns:
        return out

    for c in ["home_team_id", "away_team_id"]:
        if c in sb.columns:
            sb[c] = pd.to_numeric(sb[c], errors="coerce")

    out["home_id"] = pd.to_numeric(out["home_id"], errors="coerce")
    out["away_id"] = pd.to_numeric(out["away_id"], errors="coerce")

    # --------------------------------------------------------
    # keep only scoreboard rows relevant to the board
    # --------------------------------------------------------
    board_pairs = set()
    for h, a in zip(out["home_id"], out["away_id"]):
        if pd.notna(h) and pd.notna(a):
            board_pairs.add((int(h), int(a)))
            board_pairs.add((int(a), int(h)))

    if {"home_team_id", "away_team_id"}.issubset(sb.columns):
        sb = sb[
            sb.apply(
                lambda r: (
                    pd.notna(r["home_team_id"]) and
                    pd.notna(r["away_team_id"]) and
                    (int(r["home_team_id"]), int(r["away_team_id"])) in board_pairs
                ),
                axis=1
            )
        ].copy()

    if len(sb) == 0:
        return out

    sb = sb.drop_duplicates(subset=["game_id"], keep="last").reset_index(drop=True)

    # --------------------------------------------------------
    # fetch fresh team boxes only for relevant game_ids
    # --------------------------------------------------------
    fresh = fetch_mbb_team_boxes_for_game_ids_r(sb["game_id"].astype(str).tolist())
    if fresh is None or len(fresh) == 0:
        return out

    tb = fresh.copy()

    for c in ["game_id", "team_id", "opponent_team_id", "team_score", "opponent_team_score"]:
        if c not in tb.columns:
            tb[c] = np.nan

    tb["game_id"] = tb["game_id"].astype(str)
    tb["team_id"] = pd.to_numeric(tb["team_id"], errors="coerce")
    tb["opponent_team_id"] = pd.to_numeric(tb["opponent_team_id"], errors="coerce")
    tb["team_score"] = pd.to_numeric(tb["team_score"], errors="coerce")
    tb["opponent_team_score"] = pd.to_numeric(tb["opponent_team_score"], errors="coerce")

    # --------------------------------------------------------
    # build one row per game with home/away scores
    # --------------------------------------------------------
    if "team_home_away" in tb.columns:
        home = tb[tb["team_home_away"].astype(str).str.lower().eq("home")].copy()
        away = tb[tb["team_home_away"].astype(str).str.lower().eq("away")].copy()

        home = home[["game_id", "team_id", "team_score"]].rename(
            columns={"team_id": "home_id", "team_score": "home_score"}
        )
        away = away[["game_id", "team_id", "team_score"]].rename(
            columns={"team_id": "away_id", "team_score": "away_score"}
        )

        fresh_games = home.merge(away, on="game_id", how="inner")
    else:
        tb = tb.dropna(subset=["game_id", "team_id", "opponent_team_id", "team_score"]).copy()
        tb = tb.drop_duplicates(subset=["game_id", "team_id"], keep="last").reset_index(drop=True)

        rows = []
        for gid, g in tb.groupby("game_id", sort=False):
            if len(g) != 2:
                continue
            r1 = g.iloc[0]
            r2 = g.iloc[1]
            rows.append({
                "game_id": str(gid),
                "team1_id": int(r1["team_id"]),
                "team2_id": int(r2["team_id"]),
                "team1_score": float(r1["team_score"]),
                "team2_score": float(r2["team_score"]),
            })

        fresh_games = pd.DataFrame(rows)

    if len(fresh_games) == 0:
        return out

    fresh_games["status_type_completed"] = True
    fresh_games["status_type_state"] = "post"
    fresh_games["status_type_short_detail"] = "Final"

    if "game_id" in fresh_games.columns:
        fresh_games = fresh_games.drop_duplicates(subset=["game_id"], keep="last").reset_index(drop=True)

    # --------------------------------------------------------
    # helper: stale placeholder rows that should be overwritten
    # --------------------------------------------------------
    def _needs_score_replace(df: pd.DataFrame) -> pd.Series:
        hs = pd.to_numeric(df.get("home_score"), errors="coerce")
        aw = pd.to_numeric(df.get("away_score"), errors="coerce")

        completed = df.get("status_type_completed", pd.Series(False, index=df.index)).astype(str).str.lower().isin(["true", "1", "yes"])
        state_pre = df.get("status_type_state", pd.Series("", index=df.index)).astype(str).str.lower().isin(["pre", "scheduled"])
        state_final = df.get("status_type_state", pd.Series("", index=df.index)).astype(str).str.lower().isin(["post", "postgame", "final"])
        detail_final = df.get("status_type_short_detail", pd.Series("", index=df.index)).astype(str).str.contains("Final", case=False, na=False)

        has_real_final = completed | state_final | detail_final
        placeholder_zero = hs.fillna(-999).eq(0) & aw.fillna(-999).eq(0) & state_pre & (~has_real_final)
        missing_scores = hs.isna() | aw.isna()

        return missing_scores | placeholder_zero

    # --------------------------------------------------------
    # exact game_id match first
    # --------------------------------------------------------
    if "game_id" in out.columns and "game_id" in fresh_games.columns and {"home_id", "away_id", "home_score", "away_score"}.issubset(fresh_games.columns):
        tmp = fresh_games[[
            "game_id", "home_score", "away_score",
            "status_type_completed", "status_type_state", "status_type_short_detail"
        ]].copy()

        tmp["game_id"] = tmp["game_id"].astype(str)
        out["game_id"] = out["game_id"].astype(str)

        out = out.merge(tmp, on="game_id", how="left", suffixes=("", "_fresh_gid"))
        replace_mask = _needs_score_replace(out)

        for c in ["home_score", "away_score", "status_type_completed", "status_type_state", "status_type_short_detail"]:
            fc = f"{c}_fresh_gid"
            if fc in out.columns:
                out.loc[replace_mask, c] = np.where(
                    out.loc[replace_mask, fc].notna(),
                    out.loc[replace_mask, fc],
                    out.loc[replace_mask, c]
                )

        out = out.drop(columns=[c for c in out.columns if c.endswith("_fresh_gid")], errors="ignore")

    # --------------------------------------------------------
    # fallback by oriented home/away IDs
    # --------------------------------------------------------
    replace_mask = _needs_score_replace(out)
    if replace_mask.any() and {"home_id", "away_id", "home_score", "away_score"}.issubset(fresh_games.columns):
        tmp = out.loc[replace_mask].copy()

        fg = fresh_games[[
            "home_id", "away_id", "home_score", "away_score",
            "status_type_completed", "status_type_state", "status_type_short_detail"
        ]].copy()

        fg["home_id"] = pd.to_numeric(fg["home_id"], errors="coerce")
        fg["away_id"] = pd.to_numeric(fg["away_id"], errors="coerce")
        fg = fg.drop_duplicates(subset=["home_id", "away_id"], keep="last").reset_index(drop=True)

        tmp = tmp.merge(
            fg,
            on=["home_id", "away_id"],
            how="left",
            suffixes=("", "_fresh_id")
        )

        for c in ["home_score", "away_score", "status_type_completed", "status_type_state", "status_type_short_detail"]:
            fc = f"{c}_fresh_id"
            if fc in tmp.columns:
                tmp[c] = np.where(tmp[fc].notna(), tmp[fc], tmp[c])

        out.loc[replace_mask, [
            "home_score", "away_score",
            "status_type_completed", "status_type_state", "status_type_short_detail"
        ]] = tmp[[
            "home_score", "away_score",
            "status_type_completed", "status_type_state", "status_type_short_detail"
        ]].values

    # final hard dedupe
    dedupe_cols = [c for c in ["game_id", "away_team", "home_team", "game_dt_et"] if c in out.columns]
    if dedupe_cols:
        out = out.drop_duplicates(subset=dedupe_cols, keep="last").reset_index(drop=True)

    return out

# ============================================================
# CELL 14: ACCURACY METRICS
# ============================================================

def compute_daily_accuracy(board: pd.DataFrame) -> dict:
    if board is None or len(board) == 0:
        return {}

    g = board.copy()
    g["final_score"] = g.get("final_score", "").astype(str)
    g = g[g["final_score"].str.strip().ne("")]
    if len(g) == 0:
        return {}

    def _extract_pts(s):
        nums = re.findall(r"(\d+)", str(s))
        return (float(nums[-2]), float(nums[-1])) if len(nums) >= 2 else (np.nan, np.nan)

    pts = g["final_score"].map(_extract_pts)
    g["away_pts"] = [p[0] for p in pts]
    g["home_pts"] = [p[1] for p in pts]
    g = g.dropna(subset=["away_pts", "home_pts"])
    if len(g) == 0:
        return {}

    g["home_margin_actual"] = g["home_pts"] - g["away_pts"]

    # -------------------------------------------------
    # Winner accuracy
    # -------------------------------------------------
    g["actual_winner"] = np.where(
        g["home_pts"] > g["away_pts"], g["home_team"],
        np.where(g["away_pts"] > g["home_pts"], g["away_team"], "PUSH")
    )

    pick_col = "winner_pick" if "winner_pick" in g.columns else None
    winner_acc = np.nan
    if pick_col:
        w = g[g["actual_winner"] != "PUSH"]
        if len(w):
            winner_acc = float(
                (w[pick_col].map(canonical_team) == w["actual_winner"].map(canonical_team)).mean()
            )

    # -------------------------------------------------
    # TRUE spread accuracy:
    # predicted spread must actually cover
    # -------------------------------------------------
    spread_acc = np.nan
    if "pred_margin_home" in g.columns:
        pm = pd.to_numeric(g["pred_margin_home"], errors="coerce")
        am = pd.to_numeric(g["home_margin_actual"], errors="coerce")

        valid = pm.notna() & am.notna()
        if valid.any():
            # correct if actual margin beats predicted margin in same direction
            spread_correct = np.where(
                pm[valid] > 0,
                am[valid] > pm[valid],     # home favored by x must win by MORE than x
                np.where(
                    pm[valid] < 0,
                    am[valid] < pm[valid], # away favored by x must win by MORE than x
                    am[valid] == 0         # PK / exact zero case
                )
            )
            spread_acc = float(np.mean(spread_correct))

    # -------------------------------------------------
    # Margin MAE
    # -------------------------------------------------
    margin_mae = np.nan
    within_5 = np.nan
    if "pred_margin_home" in g.columns:
        pm = pd.to_numeric(g["pred_margin_home"], errors="coerce")
        am = pd.to_numeric(g["home_margin_actual"], errors="coerce")
        v = pm.notna() & am.notna()
        if v.any():
            err = (pm[v] - am[v]).abs()
            margin_mae = float(err.mean())
            within_5 = float((err <= 5).mean())

    # -------------------------------------------------
    # ATS vs Vegas
    # -------------------------------------------------
    ats_acc, ats_games = np.nan, 0
    if "rw_spread_home" in g.columns:
        rg = g.copy()
        rg["rw_spread_home"] = pd.to_numeric(rg["rw_spread_home"], errors="coerce")
        rg["pred_margin_home"] = pd.to_numeric(rg["pred_margin_home"], errors="coerce")
        rg["home_margin_actual"] = pd.to_numeric(rg["home_margin_actual"], errors="coerce")

        rg = rg.dropna(subset=["rw_spread_home", "pred_margin_home", "home_margin_actual"])
        if len(rg) > 0:
            rg["edge"] = rg["pred_margin_home"] - rg["rw_spread_home"]
            rg["pick"] = np.where(rg["edge"] > 0, "HOME",
                           np.where(rg["edge"] < 0, "AWAY", "NO_BET"))
            rg = rg[rg["pick"] != "NO_BET"]

            if len(rg) > 0:
                rg["cover_margin"] = rg["home_margin_actual"] - rg["rw_spread_home"]

                # HOME pick wins if actual home margin > vegas spread
                # AWAY pick wins if actual home margin < vegas spread
                ats_result = np.where(
                    rg["pick"] == "HOME",
                    rg["cover_margin"] > 0,
                    rg["cover_margin"] < 0
                )

                pushes = rg["cover_margin"] == 0
                ats_result = ats_result[~pushes]

                if len(ats_result) > 0:
                    ats_acc = float(np.mean(ats_result))
                    ats_games = int(len(ats_result))

    return {
        "games_graded": len(g),
        "winner_acc": winner_acc,
        "spread_side_acc": spread_acc,
        "margin_mae": margin_mae,
        "within_5": within_5,
        "ats_acc": ats_acc,
        "ats_games": ats_games,
    }


def render_accuracy_banner(acc: dict, date_et=None) -> str:
    def fmt(x, pct=True):
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return "—"
        return f"{x*100:.1f}%" if pct else f"{x:.2f}"

    label = f"📅 {date_et}" if date_et else "📈 Season-to-Date"
    return f"""
    <div style="background:#0d0d0d; border:1px solid #2a2a2a; border-radius:10px;
                padding:10px 14px; margin:0 0 10px 0; color:#EEE; font-size:13px;
                display:flex; gap:16px; flex-wrap:wrap; align-items:center;">
      <div style="font-weight:800; color:#00BFFF;">{label}</div>
      <div><span style="color:#AAA;">Games graded:</span> <b>{acc.get('games_graded',0)}</b></div>
      <div><span style="color:#AAA;">Winner acc:</span> <b>{fmt(acc.get('winner_acc'))}</b></div>
      <div><span style="color:#AAA;">Spread acc:</span> <b>{fmt(acc.get('spread_side_acc'))}</b></div>
      <div><span style="color:#AAA;">Margin MAE:</span> <b>{fmt(acc.get('margin_mae'), pct=False)}</b></div>
      <div><span style="color:#AAA;">Within 5:</span> <b>{fmt(acc.get('within_5'))}</b></div>
      <div><span style="color:#AAA;">ATS vs Vegas:</span> <b>{fmt(acc.get('ats_acc'))}</b>
        <span style="color:#555;">({acc.get('ats_games',0)} bets)</span></div>
    </div>"""




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Paths configured
🚀 Running startup retrain check...
📂 Loading team box history...
  team_box rows: 81,282  |  schedule rows: 6,333
📊 Building rolling snapshots + Elo...
⏭️  Model is fresh (last fit: 2026-03-10T22:34:26.993715). Loading from disk.
✅ Models loaded from disk
✅ Scheduler complete. All globals ready.


# ACCURACY

In [3]:
SEASON_ACC_CACHE_PATH = os.path.join(MODEL_DIR, "season_accuracy_by_week.json")

def _load_season_acc_cache() -> dict:
    if os.path.exists(SEASON_ACC_CACHE_PATH):
        try:
            with open(SEASON_ACC_CACHE_PATH, "r") as f:
                return json.load(f)
        except Exception:
            pass
    return {}


def _save_season_acc_cache(cache: dict):
    with open(SEASON_ACC_CACHE_PATH, "w") as f:
        json.dump(cache, f, indent=2, default=str)


def _safe_pct_mean(numer, denom):
    if denom == 0:
        return np.nan
    return float(numer) / float(denom)


def _week_key_from_date(d):
    ts = pd.Timestamp(d)
    iso = ts.isocalendar()
    return f"{int(iso.year)}-W{int(iso.week):02d}"


def _combine_accuracy_dicts(acc_list: list) -> dict:
    """
    Combines weekly cached accuracy stats into one season-to-date snapshot.
    """
    if not acc_list:
        return {}

    games_graded = 0

    winner_num = 0
    winner_den = 0

    spread_num = 0
    spread_den = 0

    within5_num = 0
    within5_den = 0

    ats_num = 0
    ats_den = 0

    mae_weighted_sum = 0.0
    mae_den = 0

    for a in acc_list:
        gg = int(a.get("games_graded", 0) or 0)
        games_graded += gg

        if a.get("winner_correct") is not None and a.get("winner_total") is not None:
            winner_num += int(a.get("winner_correct", 0) or 0)
            winner_den += int(a.get("winner_total", 0) or 0)

        if a.get("spread_correct") is not None and a.get("spread_total") is not None:
            spread_num += int(a.get("spread_correct", 0) or 0)
            spread_den += int(a.get("spread_total", 0) or 0)

        if a.get("within5_correct") is not None and a.get("within5_total") is not None:
            within5_num += int(a.get("within5_correct", 0) or 0)
            within5_den += int(a.get("within5_total", 0) or 0)

        if a.get("ats_correct") is not None and a.get("ats_total") is not None:
            ats_num += int(a.get("ats_correct", 0) or 0)
            ats_den += int(a.get("ats_total", 0) or 0)

        if a.get("margin_mae_sum") is not None and a.get("margin_mae_n") is not None:
            mae_weighted_sum += float(a.get("margin_mae_sum", 0.0) or 0.0)
            mae_den += int(a.get("margin_mae_n", 0) or 0)

    return {
        "games_graded": games_graded,
        "winner_acc": _safe_pct_mean(winner_num, winner_den),
        "spread_side_acc": _safe_pct_mean(spread_num, spread_den),
        "margin_mae": (mae_weighted_sum / mae_den) if mae_den > 0 else np.nan,
        "within_5": _safe_pct_mean(within5_num, within5_den),
        "ats_acc": _safe_pct_mean(ats_num, ats_den),
        "ats_games": ats_den,
    }


def compute_daily_accuracy_detailed(board: pd.DataFrame) -> dict:
    if board is None or len(board) == 0:
        return {
            "games_graded": 0,
            "winner_correct": 0, "winner_total": 0,
            "spread_correct": 0, "spread_total": 0,
            "margin_mae_sum": 0.0, "margin_mae_n": 0,
            "within5_correct": 0, "within5_total": 0,
            "ats_correct": 0, "ats_total": 0,
        }

    g = board.copy()
    g["final_score"] = g.get("final_score", "").astype(str)
    g = g[g["final_score"].str.strip().ne("")]
    if len(g) == 0:
        return {
            "games_graded": 0,
            "winner_correct": 0, "winner_total": 0,
            "spread_correct": 0, "spread_total": 0,
            "margin_mae_sum": 0.0, "margin_mae_n": 0,
            "within5_correct": 0, "within5_total": 0,
            "ats_correct": 0, "ats_total": 0,
        }

    def _extract_pts(s):
        nums = re.findall(r"(\d+)", str(s))
        return (float(nums[-2]), float(nums[-1])) if len(nums) >= 2 else (np.nan, np.nan)

    pts = g["final_score"].map(_extract_pts)
    g["away_pts"] = [p[0] for p in pts]
    g["home_pts"] = [p[1] for p in pts]
    g = g.dropna(subset=["away_pts", "home_pts"])

    if len(g) == 0:
        return {
            "games_graded": 0,
            "winner_correct": 0, "winner_total": 0,
            "spread_correct": 0, "spread_total": 0,
            "margin_mae_sum": 0.0, "margin_mae_n": 0,
            "within5_correct": 0, "within5_total": 0,
            "ats_correct": 0, "ats_total": 0,
        }

    g["home_margin_actual"] = g["home_pts"] - g["away_pts"]

    # winner
    g["actual_winner"] = np.where(
        g["home_pts"] > g["away_pts"], g["home_team"],
        np.where(g["away_pts"] > g["home_pts"], g["away_team"], "PUSH")
    )

    winner_correct = 0
    winner_total = 0
    if "winner_pick" in g.columns:
        w = g[g["actual_winner"] != "PUSH"].copy()
        if len(w):
            winner_correct = int((w["winner_pick"].map(canonical_team) == w["actual_winner"].map(canonical_team)).sum())
            winner_total = int(len(w))

    # true spread cover accuracy
    spread_correct = 0
    spread_total = 0
    margin_mae_sum = 0.0
    margin_mae_n = 0
    within5_correct = 0
    within5_total = 0

    if "pred_margin_home" in g.columns:
        pm = pd.to_numeric(g["pred_margin_home"], errors="coerce")
        am = pd.to_numeric(g["home_margin_actual"], errors="coerce")
        valid = pm.notna() & am.notna()

        if valid.any():
            pmv = pm[valid]
            amv = am[valid]

            spread_result = np.where(
                pmv > 0,
                amv > pmv,
                np.where(pmv < 0, amv < pmv, amv == 0)
            )

            spread_correct = int(np.sum(spread_result))
            spread_total = int(len(spread_result))

            err = (pmv - amv).abs()
            margin_mae_sum = float(err.sum())
            margin_mae_n = int(len(err))
            within5_correct = int((err <= 5).sum())
            within5_total = int(len(err))

    # ats vs vegas
    ats_correct = 0
    ats_total = 0
    if "rw_spread_home" in g.columns and "pred_margin_home" in g.columns:
        rg = g.copy()
        rg["rw_spread_home"] = pd.to_numeric(rg["rw_spread_home"], errors="coerce")
        rg["pred_margin_home"] = pd.to_numeric(rg["pred_margin_home"], errors="coerce")
        rg["home_margin_actual"] = pd.to_numeric(rg["home_margin_actual"], errors="coerce")
        rg = rg.dropna(subset=["rw_spread_home", "pred_margin_home", "home_margin_actual"])

        if len(rg):
            rg["edge"] = rg["pred_margin_home"] - rg["rw_spread_home"]
            rg["pick"] = np.where(rg["edge"] > 0, "HOME", np.where(rg["edge"] < 0, "AWAY", "NO_BET"))
            rg = rg[rg["pick"] != "NO_BET"]

            if len(rg):
                rg["cover_margin"] = rg["home_margin_actual"] - rg["rw_spread_home"]
                pushes = rg["cover_margin"] == 0
                rg = rg[~pushes].copy()

                if len(rg):
                    ats_result = np.where(rg["pick"] == "HOME", rg["cover_margin"] > 0, rg["cover_margin"] < 0)
                    ats_correct = int(np.sum(ats_result))
                    ats_total = int(len(ats_result))

    return {
        "games_graded": int(len(g)),
        "winner_correct": winner_correct,
        "winner_total": winner_total,
        "spread_correct": spread_correct,
        "spread_total": spread_total,
        "margin_mae_sum": margin_mae_sum,
        "margin_mae_n": margin_mae_n,
        "within5_correct": within5_correct,
        "within5_total": within5_total,
        "ats_correct": ats_correct,
        "ats_total": ats_total,
    }


def build_or_update_season_accuracy_cache():
    """
    Updates weekly cached season accuracy.
    Only computes weeks not already saved.
    """
    cache = _load_season_acc_cache()

    s = schedule_cur.copy()
    s["game_date_time"] = pd.to_datetime(s["game_date_time"], errors="coerce", utc=True)
    s["game_dt_et"] = s["game_date_time"].dt.tz_convert("America/New_York").dt.tz_localize(None)
    s = s.dropna(subset=["game_dt_et"]).copy()

    today_et = pd.Timestamp.now(tz="America/New_York").date()
    s = s[s["game_dt_et"].dt.date <= today_et].copy()

    all_dates = sorted(s["game_dt_et"].dt.date.unique())
    if not all_dates:
        return {}

    # group dates by ISO week
    weeks = {}
    for d in all_dates:
        wk = _week_key_from_date(d)
        weeks.setdefault(wk, []).append(d)

    updated = False

    for wk, dates_in_week in weeks.items():
        if wk in cache:
            continue

        boards = []
        for d in sorted(dates_in_week):
            try:
                b = build_board_for_date(
                    date_et=d,
                    schedule_df=schedule_cur,
                    team_snaps=team_snaps,
                    roll_cols=roll_cols,
                    spread_booster=spread_booster,
                    winner_booster=winner_booster,
                    lgb_spread=lgb_spread,
                    iso=iso,
                    imp=imp,
                    feature_cols=feature_cols,
                    elo_snap=elo_snap,
                    verbose=False,
                )
                if b is not None and len(b) > 0:
                    b = _attach_rotowire_to_board(b, d)
                    boards.append(b)
            except Exception:
                continue

        if boards:
            week_board = pd.concat(boards, ignore_index=True)
            week_board = week_board.drop_duplicates(
                subset=[c for c in ["game_id", "away_team", "home_team", "game_dt_et"] if c in week_board.columns],
                keep="last"
            )
            cache[wk] = compute_daily_accuracy_detailed(week_board)
        else:
            cache[wk] = {
                "games_graded": 0,
                "winner_correct": 0, "winner_total": 0,
                "spread_correct": 0, "spread_total": 0,
                "margin_mae_sum": 0.0, "margin_mae_n": 0,
                "within5_correct": 0, "within5_total": 0,
                "ats_correct": 0, "ats_total": 0,
            }

        updated = True
        print(f"✅ Cached season week: {wk}")

    if updated:
        _save_season_acc_cache(cache)

    return cache


def load_cached_season_accuracy_snapshot() -> dict:
    cache = _load_season_acc_cache()
    if not cache:
        return {}
    week_keys = sorted(cache.keys())
    acc_list = [cache[k] for k in week_keys]
    return _combine_accuracy_dicts(acc_list)

# DASHBOARD

In [4]:
# @title
# ============================================================
# CELL 15: DASHBOARD (PATCHED)
# ============================================================

LAST_BOARD = None
LAST_DATE  = None

SEASON_ACC = None
SEASON_ACC_DATE = None


# --- Widgets ---
date_picker     = widgets.DatePicker(description="Slate (ET):")
refresh_btn     = widgets.Button(description="🔄 Refresh", button_style="primary")
retrain_btn     = widgets.Button(description="🔁 Force Retrain", button_style="warning")
min_conf        = widgets.FloatSlider(description="Min Conf", min=0.50, max=0.95, step=0.01, value=0.60, readout_format=".2f")
min_abs_margin  = widgets.FloatSlider(description="Min |Margin|", min=0.0, max=20.0, step=0.5, value=0.0)
side_filter     = widgets.Dropdown(description="Side", options=[("All","all"),("Favorites","fav"),("Dogs","dog")], value="all")
neutral_only    = widgets.Checkbox(description="Neutral only", value=False)
show_inj        = widgets.Checkbox(description="Show injuries", value=True)
show_rw_missing = widgets.Checkbox(description="Show RW missing", value=False)
search_box      = widgets.Text(description="Search", placeholder="team name...")
max_rows        = widgets.IntSlider(description="Rows", min=10, max=300, step=10, value=80)

out         = widgets.Output()
season_banner_html = widgets.HTML()
daily_banner_html = widgets.HTML()

date_picker.value = pd.Timestamp.now(tz="America/New_York").date()


def _fmt_int_or_blank(v):
    try:
        return str(int(float(v)))
    except Exception:
        return ""


def _fmt_spread_abs(v):
    v = abs(float(v))
    return str(int(v)) if v == int(v) else f"{v:.1f}"


def _win_style(t):
    return f"<span style='color:#2ECC71;font-weight:800;'>{t}</span>"


def _lose_style(t):
    return f"<span style='color:#FF6B6B;'>{t}</span>"


def data_status_note(schedule_df, team_box_df, board=None, selected_date=None):
    schedule_max = None
    team_box_max = None
    rw_rows = 0

    try:
        s = schedule_df.copy()
        s["game_date_time"] = pd.to_datetime(s["game_date_time"], errors="coerce", utc=True)
        schedule_max = s["game_date_time"].max()
        if pd.notna(schedule_max):
            schedule_max = schedule_max.tz_convert("America/New_York").tz_localize(None)
    except:
        pass

    try:
        t = team_box_df.copy()
        t["game_date_time"] = pd.to_datetime(t["game_date_time"], errors="coerce", utc=True)
        team_box_max = t["game_date_time"].max()
        if pd.notna(team_box_max):
            team_box_max = team_box_max.tz_convert("America/New_York").tz_localize(None)
    except:
        pass

    try:
        if board is not None:
            gid = board.get("game_id", pd.Series(index=board.index, dtype=object)).astype(str)
            rw_rows = int(gid.str.startswith("RW_").sum())
    except:
        pass

    selected_ts = pd.Timestamp(selected_date).normalize() if selected_date else None

    # ---------------------------------------------------
    # suppress stale warning if board already has finals
    # ---------------------------------------------------
    board_has_finals = False
    try:
        if board is not None and len(board) > 0:
            fs = board.get("final_score", pd.Series("", index=board.index)).astype(str).str.strip()
            board_has_finals = fs.ne("").any()

            # fallback check in case final_score wasn't built yet
            if not board_has_finals:
                completed = board.get("status_type_completed", pd.Series(False, index=board.index)).astype(str).str.lower().isin(["true", "1", "yes"])
                state_final = board.get("status_type_state", pd.Series("", index=board.index)).astype(str).str.lower().isin(["post", "postgame", "final"])
                detail_final = board.get("status_type_short_detail", pd.Series("", index=board.index)).astype(str).str.contains("Final", case=False, na=False)

                hs = pd.to_numeric(board.get("home_score", pd.Series(np.nan, index=board.index)), errors="coerce")
                aw = pd.to_numeric(board.get("away_score", pd.Series(np.nan, index=board.index)), errors="coerce")

                board_has_finals = ((completed | state_final | detail_final) & hs.notna() & aw.notna()).any()
    except:
        pass

    stale = False
    if (selected_ts is not None) and (team_box_max is not None) and (not board_has_finals):
        if pd.Timestamp(team_box_max).normalize() < selected_ts:
            stale = True

    warning_html = ""
    if stale:
        warning_html = f"""
        <div style="
            background:#2a0000;
            border-left:5px solid #ff4d4f;
            padding:10px;
            margin-bottom:8px;
            color:#ff6b6b;
            font-weight:700;
            font-size:13px;
        ">
        ⚠ RESULTS DATA OUTDATED<br>
        Team box only updated through: {team_box_max}<br>
        Selected slate: {selected_ts.date()}
        </div>
        """

    info_html = f"""
    <div style="
        background:#111;
        border-left:4px solid #555;
        padding:8px;
        font-size:12px;
        color:#bbb;
        margin-bottom:10px;
    ">
    Schedule data through: {schedule_max}<br>
    Team box data through: {team_box_max}<br>
    Rotowire fallback rows: {rw_rows}
    </div>
    """

    return warning_html + info_html


def render_dashboard_banner(acc: dict, date_et=None, note: str = "") -> str:
    base = render_accuracy_banner(acc, date_et=date_et)
    if note:
        base += f"<div style='color:#888; font-size:12px; margin:4px 0 8px 0;'>{note}</div>"
    return base


def df_to_html_table(df: pd.DataFrame, max_rows: int = 80) -> str:
    if df is None or len(df) == 0:
        return "<div style='color:#AAA; padding:8px;'>No games match current filters.</div>"
    df = df.head(max_rows)
    style = """
    <style>
    .pred-table { border-collapse: collapse; width:100%; font-size:12px; color:#EEE; }
    .pred-table th { background:#1a1a1a; color:#FFD700; padding:6px 8px;
                     border-bottom:2px solid #333; white-space:nowrap; }
    .pred-table td { padding:5px 8px; border-bottom:1px solid #222; white-space:nowrap; }
    .pred-table tr:hover td { background:#1f1f1f; }
    </style>"""
    return style + df.to_html(escape=False, index=False, classes="pred-table", border=0)


def _apply_filters(board: pd.DataFrame) -> pd.DataFrame:
    if board is None or len(board) == 0:
        return pd.DataFrame()

    b = board.copy()

    if "pick_conf" in b.columns:
        b = b[pd.to_numeric(b["pick_conf"], errors="coerce").fillna(0) >= float(min_conf.value)]

    if "pred_margin_home" in b.columns:
        b = b[pd.to_numeric(b["pred_margin_home"], errors="coerce").abs().fillna(0) >= float(min_abs_margin.value)]

    if neutral_only.value and "neutral_site" in b.columns:
        b = b[b["neutral_site"].astype(str).str.lower().isin(["true", "1", "yes", "y"])]

    if side_filter.value == "fav":
        fav = np.where(b["pred_margin_home"] >= 0, b["home_team"], b["away_team"])
        b = b[b["winner_pick"] == fav]
    elif side_filter.value == "dog":
        fav = np.where(b["pred_margin_home"] >= 0, b["home_team"], b["away_team"])
        b = b[b["winner_pick"] != fav]

    s = search_box.value.strip().lower()
    if s:
        b = b[
            (b["home_team"].str.lower().str.contains(s, na=False)) |
            (b["away_team"].str.lower().str.contains(s, na=False))
        ]

    return b


def _is_real_final_row(row) -> bool:
    try:
        hs = pd.to_numeric(row.get("home_score"), errors="coerce")
        aw = pd.to_numeric(row.get("away_score"), errors="coerce")
    except Exception:
        return False

    if pd.isna(hs) or pd.isna(aw):
        return False

    completed = str(row.get("status_type_completed", "")).strip().lower() in {"true", "1", "yes"}
    state_ok  = str(row.get("status_type_state", "")).strip().lower() in {"post", "postgame", "final"}
    detail_ok = "final" in str(row.get("status_type_short_detail", "")).strip().lower()

    return completed or state_ok or detail_ok


def _format_board_for_display(board: pd.DataFrame) -> pd.DataFrame:
    b = board.copy()

    for col in ["pick_conf", "p_home_win"]:
        if col in b.columns:
            b[col] = pd.to_numeric(b[col], errors="coerce").map(
                lambda v: "—" if pd.isna(v) else f"{v*100:.1f}%"
            )

    if "rw_spread_home" in b.columns:
        sp = pd.to_numeric(b["rw_spread_home"], errors="coerce")
        vt = []
        for ht, at, v in zip(b["home_team"], b["away_team"], sp):
            if pd.isna(v):
                vt.append("")
            elif abs(float(v)) < 1e-9:
                vt.append("PK")
            elif float(v) < 0:
                vt.append(f"{ht} {float(v):.1f}")
            else:
                vt.append(f"{at} -{abs(float(v)):.1f}")
        b["Vegas Spread"] = vt
    else:
        b["Vegas Spread"] = ""

    if all(c in b.columns for c in ["rw_home_ml", "rw_away_ml"]):
        hml = pd.to_numeric(b["rw_home_ml"], errors="coerce")
        aml = pd.to_numeric(b["rw_away_ml"], errors="coerce")
        mls = []

        for ht, at, wp, hv, av in zip(
            b["home_team"],
            b["away_team"],
            b.get("winner_pick", [""] * len(b)),
            hml,
            aml
        ):
            if str(wp) == str(ht) and pd.notna(hv):
                hv = int(round(float(hv)))
                mls.append(f"{ht} {'+' if hv > 0 else ''}{hv}")
            elif str(wp) == str(at) and pd.notna(av):
                av = int(round(float(av)))
                mls.append(f"{at} {'+' if av > 0 else ''}{av}")
            else:
                mls.append("")
        b["Vegas ML"] = mls
    else:
        b["Vegas ML"] = ""

    # Model Spread + Pred_vs_Final
    if "pred_margin_home" in b.columns:
        pm = pd.to_numeric(b["pred_margin_home"], errors="coerce")
        hs = pd.to_numeric(b.get("home_score", pd.Series(np.nan, index=b.index)), errors="coerce")
        aw = pd.to_numeric(b.get("away_score", pd.Series(np.nan, index=b.index)), errors="coerce")

        real_final_mask = b.apply(_is_real_final_row, axis=1)
        actual_margin = (hs - aw).where(real_final_mask, np.nan)

        def _pred_vs_final_spread(pred_margin, actual_margin):
            if pd.isna(pred_margin) or pd.isna(actual_margin):
                return ""
            if float(pred_margin) == 0 or float(actual_margin) == 0:
                return "P"
            return "W" if ((pred_margin > 0 and actual_margin > pred_margin) or
                           (pred_margin < 0 and actual_margin < pred_margin)) else "L"

        b["Pred_vs_Final"] = [
            _pred_vs_final_spread(p, a)
            for p, a in zip(pm, actual_margin)
        ]

        model_txt = []
        for ht, at, v in zip(b["home_team"], b["away_team"], pm):
            if pd.isna(v):
                model_txt.append("")
            elif abs(float(v)) < 1e-9:
                model_txt.append("PK")
            elif float(v) > 0:
                model_txt.append(f"{ht} -{_fmt_spread_abs(v)}")
            else:
                model_txt.append(f"{at} -{_fmt_spread_abs(v)}")

        def _col(txt, res):
            if res == "W":
                return f"<span style='color:#2ECC71;font-weight:800;'>{txt}</span>"
            if res == "L":
                return f"<span style='color:#FF6B6B;font-weight:800;'>{txt}</span>"
            if res == "P":
                return f"<span style='color:#FFD166;font-weight:800;'>{txt}</span>"
            return txt

        b["Model Spread"] = [_col(t, r) for t, r in zip(model_txt, b["Pred_vs_Final"])]
    else:
        b["Model Spread"] = ""
        b["Pred_vs_Final"] = ""

    finals = []
    if all(c in b.columns for c in ["home_score", "away_score"]):
        hs = pd.to_numeric(b["home_score"], errors="coerce")
        aw = pd.to_numeric(b["away_score"], errors="coerce")
        real_final_mask = b.apply(_is_real_final_row, axis=1)

        for ht, at, h, a, is_final in zip(b["home_team"], b["away_team"], hs, aw, real_final_mask):
            if not is_final or pd.isna(h) or pd.isna(a):
                finals.append("")
                continue

            h, a = float(h), float(a)
            home_t = f"{ht} {_fmt_int_or_blank(h)}"
            away_t = f"{at} {_fmt_int_or_blank(a)}"

            if h > a:
                finals.append(f"<div>{_win_style(home_t)}</div><div>{_lose_style(away_t)}</div>")
            elif a > h:
                finals.append(f"<div>{_lose_style(home_t)}</div><div>{_win_style(away_t)}</div>")
            else:
                finals.append(f"<div>{home_t}</div><div>{away_t}</div>")
    else:
        finals = [""] * len(b)

    b["Final"] = finals

    if show_inj.value and "home_injury_impact" in b.columns:
        b["home_injury_impact"] = pd.to_numeric(b["home_injury_impact"], errors="coerce").map(
            lambda v: f"{v:.0f}" if pd.notna(v) and v > 0 else ""
        )
        b["away_injury_impact"] = pd.to_numeric(b["away_injury_impact"], errors="coerce").map(
            lambda v: f"{v:.0f}" if pd.notna(v) and v > 0 else ""
        )
    else:
        b = b.drop(columns=["home_injury_impact", "away_injury_impact"], errors="ignore")

    if "rw_total" in b.columns:
        b["rw_total"] = pd.to_numeric(b["rw_total"], errors="coerce").map(
            lambda v: "" if pd.isna(v) else f"{float(v):.1f}"
        )

    rename = {
        "game_dt_et": "Time",
        "away_team": "Away",
        "home_team": "Home",
        "pick_conf": "Confidence",
        "p_home_win": "Home Win%",
        "winner_pick": "ML Pick",
        "spread_pick": "Spread Pick",
        "rw_total": "Total",
        "home_injury_impact": "Home Inj",
        "away_injury_impact": "Away Inj",
    }

    final_order = [
        "game_dt_et", "away_team", "home_team",
        "pick_conf", "p_home_win",
        "Model Spread", "winner_pick", "spread_pick",
        "Vegas Spread", "Vegas ML",
        "Final", "Pred_vs_Final",
        "rw_total",
        "home_injury_impact", "away_injury_impact",
    ]

    b = b[[c for c in final_order if c in b.columns]]
    b = b.rename(columns={k: v for k, v in rename.items() if k in b.columns})
    return b

def render_two_level_banner(season_acc: dict = None, season_label: str = "", daily_acc: dict = None, daily_label: str = "", note: str = "") -> tuple:
    season_html = ""
    daily_html = ""

    if season_acc is not None:
        season_html = render_accuracy_banner(season_acc, date_et=season_label)

    if daily_acc is not None:
        daily_html = render_accuracy_banner(daily_acc, date_et=daily_label)

    if note:
        daily_html += note

    return season_html, daily_html



# Helpers go above this line
def refresh(_=None, force_rebuild=False):
    global LAST_BOARD, LAST_DATE, SEASON_ACC, SEASON_ACC_DATE

    # build season snapshot once only
    if SEASON_ACC is None:
        SEASON_ACC = {}
        SEASON_ACC_DATE = "Season-to-Date"

    # blank initial render
    season_html, daily_html = render_two_level_banner(
        season_acc=SEASON_ACC,
        season_label=SEASON_ACC_DATE,
        daily_acc={},
        daily_label=str(date_picker.value),
        note=""
    )
    season_banner_html.value = season_html
    daily_banner_html.value = daily_html

    # fast path
    if (LAST_DATE == date_picker.value) and (LAST_BOARD is not None) and not force_rebuild:
        with out:
            clear_output(wait=True)

            b2 = _apply_filters(LAST_BOARD)
            filtered_acc = compute_daily_accuracy(b2)

            note_html = data_status_note(
                schedule_cur,
                team_box_hist,
                LAST_BOARD,
                selected_date=date_picker.value
            ) if "team_box_hist" in globals() else ""

            season_html, daily_html = render_two_level_banner(
                season_acc=SEASON_ACC,
                season_label=SEASON_ACC_DATE,
                daily_acc=filtered_acc,
                daily_label=str(date_picker.value),
                note=note_html
            )

            season_banner_html.value = season_html
            daily_banner_html.value = daily_html

            display(HTML(df_to_html_table(_format_board_for_display(b2), int(max_rows.value))))
        return

    with out:
        clear_output(wait=True)
        display(HTML("<div style='color:#AAA; padding:8px;'>⏳ Building board...</div>"))

    try:
        with suppress_stdout_stderr():
            board = build_board_for_date(
                date_et=date_picker.value,
                schedule_df=schedule_cur,
                team_snaps=team_snaps,
                roll_cols=roll_cols,
                spread_booster=spread_booster,
                winner_booster=winner_booster,
                lgb_spread=lgb_spread,
                iso=iso,
                imp=imp,
                feature_cols=feature_cols,
                elo_snap=elo_snap,
            )
    except Exception as e:
        with out:
            clear_output(wait=True)
            display(HTML(f"<div style='color:#ffb4b4;'>❌ Board error: {e}</div>"))
        return

    if board is None or len(board) == 0:
        with out:
            clear_output(wait=True)
            display(HTML("<div style='color:#AAA;'>No games found for this date.</div>"))
        return

    board = _attach_rotowire_to_board(board, date_picker.value)

    LAST_BOARD = board
    LAST_DATE = date_picker.value

    b2 = _apply_filters(board)
    filtered_acc = compute_daily_accuracy(b2)

    note_html = data_status_note(
        schedule_cur,
        team_box_hist,
        board,
        selected_date=date_picker.value
    ) if "team_box_hist" in globals() else ""

    season_html, daily_html = render_two_level_banner(
        season_acc=SEASON_ACC,
        season_label=SEASON_ACC_DATE,
        daily_acc=filtered_acc,
        daily_label=str(date_picker.value),
        note=note_html
    )

    season_banner_html.value = season_html
    daily_banner_html.value = daily_html

    with out:
        clear_output(wait=True)
        if show_rw_missing.value:
            miss = board[board.get("rw_missing_reason", pd.Series("", index=board.index)).ne("")]
            if len(miss):
                display(HTML(f"<div style='color:#AAA;'>⚠️ RW missing: {len(miss)} games</div>"))
                display(miss[["away_team", "home_team", "rw_missing_reason"]].head(20))
        display(HTML(df_to_html_table(_format_board_for_display(b2), int(max_rows.value))))


def force_retrain_clicked(_=None):
    with out:
        clear_output(wait=True)
        display(HTML("<div style='color:#FFD700;'>🔁 Forcing full model retrain...</div>"))
    check_and_retrain(force_data=False, force_model=True)
    refresh(force_rebuild=True)


refresh_btn.on_click(lambda _: refresh(force_rebuild=True))
retrain_btn.on_click(force_retrain_clicked)
date_picker.observe(lambda ch: refresh() if ch["name"] == "value" else None, names="value")

for widget in [min_conf, min_abs_margin, side_filter, neutral_only, show_inj,
               show_rw_missing, search_box, max_rows]:
    widget.observe(lambda ch: refresh() if ch["name"] == "value" else None, names="value")

# Layout
controls_row1 = widgets.HBox([date_picker, refresh_btn, retrain_btn])
controls_row2 = widgets.HBox([min_conf, min_abs_margin, side_filter])
controls_row3 = widgets.HBox([neutral_only, show_inj, show_rw_missing, search_box, max_rows])

display(controls_row1, controls_row2, controls_row3, season_banner_html, daily_banner_html, out)
refresh()


HTML(value='')

HTML(value='')

Output()

# REPORT

In [5]:
# @title
# ============================================================
# CELL 16: STATIC HIGH-CONFIDENCE REPORT
# Renders once from the current board and stays fixed
# ============================================================

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

HC_REPORT_CACHE = None
HC_REPORT_DATE = None

hc_thresh = widgets.FloatSlider(
    description="Min Conf",
    min=0.50, max=0.95, step=0.01, value=0.60,
    readout_format=".2f"
)

build_hc_btn = widgets.Button(
    description="Build Report",
    button_style="primary",
    icon="check"
)

hc_out = widgets.Output()


def _pct(x):
    return "—" if (x is None or (isinstance(x, float) and np.isnan(x))) else f"{x*100:.1f}%"


def _normalize_report_board(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    b = df.copy()

    if "home_team" not in b.columns:
        if "home_short_display_name" in b.columns:
            b["home_team"] = b["home_short_display_name"].astype(str)
        else:
            b["home_team"] = "TBD"

    if "away_team" not in b.columns:
        if "away_short_display_name" in b.columns:
            b["away_team"] = b["away_short_display_name"].astype(str)
        else:
            b["away_team"] = "TBD"

    if "confidence" not in b.columns and "pick_conf" in b.columns:
        b["confidence"] = pd.to_numeric(b["pick_conf"], errors="coerce")

    return b


def _safe_ats_vs_rotowire(g: pd.DataFrame) -> dict:
    if g is None or len(g) == 0:
        return {"ats_games": 0, "ats_acc": np.nan, "pushes": 0}

    req = {"pred_margin_home", "rw_spread_home", "home_score", "away_score"}
    if not req.issubset(g.columns):
        return {"ats_games": 0, "ats_acc": np.nan, "pushes": 0}

    x = g.copy()
    x["pred_margin_home"] = pd.to_numeric(x["pred_margin_home"], errors="coerce")
    x["rw_spread_home"] = pd.to_numeric(x["rw_spread_home"], errors="coerce")
    x["hs"] = pd.to_numeric(x["home_score"], errors="coerce")
    x["as"] = pd.to_numeric(x["away_score"], errors="coerce")
    x = x.dropna(subset=["pred_margin_home", "rw_spread_home", "hs", "as"])

    if len(x) == 0:
        return {"ats_games": 0, "ats_acc": np.nan, "pushes": 0}

    x["hma"] = x["hs"] - x["as"]
    x["edge"] = x["pred_margin_home"] - x["rw_spread_home"]
    x = x[x["edge"].abs() > 0]

    if len(x) == 0:
        return {"ats_games": 0, "ats_acc": np.nan, "pushes": 0}

    x["pick"] = np.where(x["edge"] > 0, "HOME", "AWAY")
    x["cover"] = x["hma"] + x["rw_spread_home"]

    pushes = int((x["cover"] == 0).sum())
    x = x[x["cover"] != 0]

    if len(x) == 0:
        return {"ats_games": 0, "ats_acc": np.nan, "pushes": pushes}

    w = np.where(x["pick"] == "HOME", x["cover"] > 0, x["cover"] < 0)
    return {"ats_games": int(len(x)), "ats_acc": float(np.mean(w)), "pushes": pushes}


def _winner_acc(g: pd.DataFrame) -> float:
    if g is None or len(g) == 0 or "winner_pick" not in g.columns:
        return np.nan
    req = {"winner_pick", "home_score", "away_score", "home_team", "away_team"}
    if not req.issubset(g.columns):
        return np.nan

    x = g.copy()
    x["home_score"] = pd.to_numeric(x["home_score"], errors="coerce")
    x["away_score"] = pd.to_numeric(x["away_score"], errors="coerce")
    x = x.dropna(subset=["home_score", "away_score"])

    if len(x) == 0:
        return np.nan

    actual = np.where(x["home_score"] > x["away_score"], x["home_team"], x["away_team"])
    return float((x["winner_pick"].astype(str).values == actual).mean())


def _build_static_hc_report():
    global HC_REPORT_CACHE, HC_REPORT_DATE, LAST_BOARD, LAST_DATE

    # Use the already-rendered dashboard board only.
    if "LAST_BOARD" not in globals() or LAST_BOARD is None or len(LAST_BOARD) == 0:
        return None, "No current board is available yet. Render the dashboard first."

    board = _normalize_report_board(LAST_BOARD.copy())
    report_date = LAST_DATE if "LAST_DATE" in globals() else None

    # attach RW only if not already there
    if "rw_spread_home" not in board.columns:
        try:
            board = _attach_rotowire_to_board(board, report_date)
        except Exception:
            pass

    conf = pd.to_numeric(board.get("confidence", board.get("pick_conf")), errors="coerce")
    board = board.assign(confidence=conf)

    hc = board[board["confidence"] >= float(hc_thresh.value)].copy()

    # keep it static and deduped
    dedupe_cols = [c for c in ["game_id"] if c in hc.columns]
    if not dedupe_cols:
        dedupe_cols = [c for c in ["game_dt_et", "away_team", "home_team"] if c in hc.columns]
    if dedupe_cols:
        hc = hc.drop_duplicates(subset=dedupe_cols, keep="last").reset_index(drop=True)

    HC_REPORT_CACHE = hc.copy()
    HC_REPORT_DATE = report_date
    return hc, None


def _render_static_hc_report(_=None):
    with hc_out:
        clear_output(wait=True)

        hc, err = _build_static_hc_report()
        if err is not None:
            display(HTML(f"<div style='color:#ffb4b4;'>❌ {err}</div>"))
            return

        if hc is None or len(hc) == 0:
            display(HTML("<div style='color:#AAA;'>No high-confidence games for the current board.</div>"))
            return

        graded = hc.copy()
        graded["home_score"] = pd.to_numeric(graded.get("home_score"), errors="coerce")
        graded["away_score"] = pd.to_numeric(graded.get("away_score"), errors="coerce")
        graded = graded[graded["home_score"].notna() & graded["away_score"].notna()].copy()

        win_acc = _winner_acc(graded)
        ats = _safe_ats_vs_rotowire(graded)

        display(HTML(
            f"""
            <div style='padding:10px 0 14px 0; line-height:1.7;'>
              <div style='font-size:18px; font-weight:700;'>📄 High-Confidence Report</div>
              <div>Date (ET): <b>{HC_REPORT_DATE}</b></div>
              <div>Threshold: <b>{hc_thresh.value:.2f}</b></div>
              <div>Rows: <b>{len(hc)}</b></div>
              <div>Graded finals: <b>{len(graded)}</b></div>
              <div>Winner acc: <b>{_pct(win_acc)}</b></div>
              <div>ATS vs RW: <b>{_pct(ats['ats_acc'])}</b> ({ats['ats_games']} bets, {ats['pushes']} pushes)</div>
              <div style='color:#AAA;'>Static snapshot from the current rendered slate only. Does not rebuild season history.</div>
            </div>
            """
        ))

        show_cols = [
            "game_dt_et", "away_team", "home_team",
            "confidence", "p_home_win", "pred_margin_home",
            "winner_pick", "spread_pick",
            "rw_spread_home", "rw_home_ml", "rw_away_ml", "rw_total",
            "final_score",
            "home_injury_impact", "away_injury_impact"
        ]
        show_cols = [c for c in show_cols if c in hc.columns]

        display(hc[show_cols].sort_values(["game_dt_et", "home_team", "away_team"]).reset_index(drop=True))


build_hc_btn.on_click(_render_static_hc_report)

display(widgets.HBox([hc_thresh, build_hc_btn]), hc_out)

Output()